In [96]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_seq_items', 10000)
pd.set_option("display.width", 200)


In [3]:
from pathlib import Path

CPP_DIR = Path(
    r"C:\Users\sruddy1\Documents\satuRnPaper\double_binomial_cpp"
)

cpp_code = r'''
#include <pybind11/pybind11.h>

#include <algorithm>
#include <cmath>
#include <limits>
#include <stdexcept>
#include <string>

namespace py = pybind11;

constexpr double PI = 3.141592653589793238462643383279502884;
constexpr double SQRT2 = 1.414213562373095048801688724209698079;
constexpr double LOG_SQRT_2PI = 0.918938533204672741780329736405617640;


void validate_parameters(int m, double p, int y) {
    if (m < 1) {
        throw std::invalid_argument("m must be >= 1.");
    }

    if (!(p > 0.0 && p < 1.0)) {
        throw std::invalid_argument("p must lie strictly in (0, 1).");
    }

    if (y < 0 || y > m) {
        throw std::invalid_argument("y must lie in {0, ..., m}.");
    }
}


double log_choose(int m, int y) {
    return (
        std::lgamma(static_cast<double>(m) + 1.0)
        - std::lgamma(static_cast<double>(y) + 1.0)
        - std::lgamma(static_cast<double>(m - y) + 1.0)
    );
}


double log_binomial_pmf(int m, double p, int y) {
    validate_parameters(m, p, y);

    return (
        log_choose(m, y)
        + static_cast<double>(y) * std::log(p)
        + static_cast<double>(m - y) * std::log1p(-p)
    );
}


double log_saturated_binomial_pmf(int m, int y) {
    if (m < 1) {
        throw std::invalid_argument("m must be >= 1.");
    }

    if (y < 0 || y > m) {
        throw std::invalid_argument("y must lie in {0, ..., m}.");
    }

    double result = log_choose(m, y);

    if (y > 0) {
        result += (
            static_cast<double>(y)
            * std::log(
                static_cast<double>(y)
                / static_cast<double>(m)
            )
        );
    }

    if (y < m) {
        result += (
            static_cast<double>(m - y)
            * std::log(
                static_cast<double>(m - y)
                / static_cast<double>(m)
            )
        );
    }

    return result;
}


double binomial_deviance(int m, double p, int y) {
    validate_parameters(m, p, y);

    const double yd = static_cast<double>(y);
    const double md = static_cast<double>(m);

    const double mean_success = md * p;
    const double mean_failure = md * (1.0 - p);

    double value = 0.0;

    if (y > 0) {
        value += (
            yd
            * std::log(
                yd
                / mean_success
            )
        );
    }

    if (y < m) {
        const double failures = md - yd;

        value += (
            failures
            * std::log(
                failures
                / mean_failure
            )
        );
    }

    value *= 2.0;

    if (value < 0.0 && value > -1e-12) {
        value = 0.0;
    }

    return std::max(value, 0.0);
}


double signed_root_deviance(int m, double p, int y) {
    const double difference = (
        static_cast<double>(y)
        - static_cast<double>(m) * p
    );

    if (difference == 0.0) {
        return 0.0;
    }

    const double root = std::sqrt(
        binomial_deviance(m, p, y)
    );

    return (
        difference > 0.0
        ? root
        : -root
    );
}


double log_normal_cdf(double x) {
    if (std::isinf(x)) {
        return (
            x > 0.0
            ? 0.0
            : -std::numeric_limits<double>::infinity()
        );
    }

    /*
    For the far left tail use the Mills-ratio expansion

        Phi(-t)
            = phi(t)/t
              * [1 - 1/t^2 + 3/t^4 - 15/t^6 + ...].

    This avoids erfc underflow for the extreme deviance coordinates that
    occur when p is very close to 0 or 1.
    */

    if (x < -10.0) {
        const double t = -x;
        const double inv_t2 = 1.0 / (t * t);

        double series = 1.0;
        double term = 1.0;

        for (int k = 1; k <= 40; ++k) {
            term *= (
                -static_cast<double>(2 * k - 1)
                * inv_t2
            );

            series += term;

            if (
                std::abs(term)
                <= 1e-18 * std::abs(series)
            ) {
                break;
            }
        }

        if (!(series > 0.0)) {
            throw std::runtime_error(
                "Mills-ratio expansion became nonpositive."
            );
        }

        return (
            -0.5 * t * t
            - std::log(t)
            - LOG_SQRT_2PI
            + std::log(series)
        );
    }

    const double probability = (
        0.5
        * std::erfc(
            -x / SQRT2
        )
    );

    if (probability <= 0.0) {
        throw std::runtime_error(
            "Normal CDF underflowed unexpectedly."
        );
    }

    return std::log(probability);
}


double log_diff_exp(double log_x, double log_y) {
    if (std::isinf(log_y) && log_y < 0.0) {
        return log_x;
    }

    double delta = (
        log_y - log_x
    );

    if (delta >= 0.0) {
        delta = std::nextafter(
            0.0,
            -std::numeric_limits<double>::infinity()
        );
    }

    return (
        log_x
        + std::log(
            -std::expm1(delta)
        )
    );
}


double log_normal_interval_probability(
    double lower,
    double upper
) {
    if (!(lower < upper)) {
        throw std::invalid_argument(
            "Normal interval must satisfy lower < upper."
        );
    }

    if (std::isinf(lower) && lower < 0.0) {
        return log_normal_cdf(upper);
    }

    if (std::isinf(upper) && upper > 0.0) {
        return log_normal_cdf(-lower);
    }

    if (upper <= 0.0) {
        return log_diff_exp(
            log_normal_cdf(upper),
            log_normal_cdf(lower)
        );
    }

    if (lower >= 0.0) {
        return log_diff_exp(
            log_normal_cdf(-lower),
            log_normal_cdf(-upper)
        );
    }

    const double cdf_upper = (
        0.5
        * std::erfc(
            -upper / SQRT2
        )
    );

    const double cdf_lower = (
        0.5
        * std::erfc(
            -lower / SQRT2
        )
    );

    const double probability = (
        cdf_upper - cdf_lower
    );

    if (!(probability > 0.0)) {
        throw std::runtime_error(
            "Central Normal interval probability became nonpositive."
        );
    }

    return std::log(probability);
}


std::pair<double, double> deviance_cell_bounds(
    int m,
    double p,
    int y
) {
    validate_parameters(m, p, y);

    const double z_current = signed_root_deviance(
        m,
        p,
        y
    );

    double lower = -std::numeric_limits<double>::infinity();
    double upper = std::numeric_limits<double>::infinity();

    if (y > 0) {
        const double z_left = signed_root_deviance(
            m,
            p,
            y - 1
        );

        lower = (
            0.5
            * (
                z_left
                + z_current
            )
        );
    }

    if (y < m) {
        const double z_right = signed_root_deviance(
            m,
            p,
            y + 1
        );

        upper = (
            0.5
            * (
                z_current
                + z_right
            )
        );
    }

    return {
        lower,
        upper
    };
}


double source_carrier(
    int m,
    double p,
    int y
) {
    const auto bounds = deviance_cell_bounds(
        m,
        p,
        y
    );

    const double log_interval = (
        log_normal_interval_probability(
            bounds.first,
            bounds.second
        )
    );

    return (
        log_binomial_pmf(
            m,
            p,
            y
        )
        - log_interval
    );
}


double target_carrier(
    int m,
    double p,
    double theta,
    int y
) {
    if (!(theta > 0.0)) {
        throw std::invalid_argument(
            "theta must be positive."
        );
    }

    const auto bounds = deviance_cell_bounds(
        m,
        p,
        y
    );

    const double root_theta = std::sqrt(theta);

    const double log_interval = (
        log_normal_interval_probability(
            root_theta * bounds.first,
            root_theta * bounds.second
        )
    );

    const double log_weight = (
        theta
        * log_binomial_pmf(
            m,
            p,
            y
        )
        + (
            1.0 - theta
        )
        * log_saturated_binomial_pmf(
            m,
            y
        )
    );

    return (
        log_weight
        - log_interval
    );
}


py::dict stage_a_values(
    int m,
    double p,
    double theta,
    int y
) {
    const auto bounds = deviance_cell_bounds(
        m,
        p,
        y
    );

    py::dict result;

    result["m"] = m;
    result["p"] = p;
    result["theta"] = theta;
    result["y"] = y;

    result["deviance"] = binomial_deviance(
        m,
        p,
        y
    );

    result["signed_root_deviance"] = signed_root_deviance(
        m,
        p,
        y
    );

    result["cell_lower"] = bounds.first;
    result["cell_upper"] = bounds.second;

    result["log_binomial_pmf"] = log_binomial_pmf(
        m,
        p,
        y
    );

    result["log_saturated_binomial_pmf"] = (
        log_saturated_binomial_pmf(
            m,
            y
        )
    );

    result["source_carrier"] = source_carrier(
        m,
        p,
        y
    );

    result["target_carrier"] = target_carrier(
        m,
        p,
        theta,
        y
    );

    return result;
}


std::string cpp_version() {
#if __cplusplus >= 202302L
    return "C++23";
#elif __cplusplus >= 202002L
    return "C++20";
#elif __cplusplus >= 201703L
    return "C++17";
#elif __cplusplus >= 201402L
    return "C++14";
#else
    return "pre-C++14";
#endif
}


PYBIND11_MODULE(fast_db_stage_a, m) {
    m.doc() = (
        "Stage-A compiled mathematical primitives "
        "for the double-binomial transport sampler"
    );

    m.def(
        "cpp_version",
        &cpp_version
    );

    m.def(
        "binomial_deviance",
        &binomial_deviance,
        py::arg("m"),
        py::arg("p"),
        py::arg("y")
    );

    m.def(
        "signed_root_deviance",
        &signed_root_deviance,
        py::arg("m"),
        py::arg("p"),
        py::arg("y")
    );

    m.def(
        "source_carrier",
        &source_carrier,
        py::arg("m"),
        py::arg("p"),
        py::arg("y")
    );

    m.def(
        "target_carrier",
        &target_carrier,
        py::arg("m"),
        py::arg("p"),
        py::arg("theta"),
        py::arg("y")
    );

    m.def(
        "stage_a_values",
        &stage_a_values,
        py::arg("m"),
        py::arg("p"),
        py::arg("theta"),
        py::arg("y")
    );
}
'''

setup_code = r'''
from setuptools import setup
from pybind11.setup_helpers import Pybind11Extension, build_ext


ext_modules = [
    Pybind11Extension(
        "fast_db_stage_a",
        ["fast_db_stage_a.cpp"],
        cxx_std=17,
        extra_compile_args=[
            "/O2",
            "/Zc:__cplusplus",
            "/fp:precise",
        ],
    ),
]


setup(
    name="fast_db_stage_a",
    version="0.0.1",
    ext_modules=ext_modules,
    cmdclass={
        "build_ext": build_ext,
    },
)
'''

(CPP_DIR / "fast_db_stage_a.cpp").write_text(
    cpp_code,
    encoding="utf-8",
)

(CPP_DIR / "setup_stage_a.py").write_text(
    setup_code,
    encoding="utf-8",
)

print("Created:")
print(CPP_DIR / "fast_db_stage_a.cpp")
print(CPP_DIR / "setup_stage_a.py")


Created:
C:\Users\sruddy1\Documents\satuRnPaper\double_binomial_cpp\fast_db_stage_a.cpp
C:\Users\sruddy1\Documents\satuRnPaper\double_binomial_cpp\setup_stage_a.py


In [4]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_a.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

print("\nreturn code:", result.returncode)

STDOUT:
running build_ext
building 'fast_db_stage_a' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tp

In [5]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(CPP_DIR),
    )

import fast_db_stage_a

print("C++ version:", fast_db_stage_a.cpp_version())

print(
    fast_db_stage_a.stage_a_values(
        m=100,
        p=0.2,
        theta=0.5,
        y=20,
    )
)


C++ version: C++17
{'m': 100, 'p': 0.2, 'theta': 0.5, 'y': 20, 'deviance': 0.0, 'signed_root_deviance': 0.0, 'cell_lower': -0.12580048990299056, 'cell_upper': 0.12423697761964274, 'log_binomial_pmf': -2.309607544702505, 'log_saturated_binomial_pmf': -2.3096075447025015, 'source_carrier': -0.0019219709102125115, 'target_carrier': 0.3433510286716266}


In [7]:
import numpy as np

from scipy.special import gammaln
from scipy.special import log_ndtr
from scipy.special import ndtr


def python_logdiffexp(log_x, log_y):
    delta = log_y - log_x

    if delta >= 0.0:
        delta = np.nextafter(
            0.0,
            -np.inf,
        )

    return (
        log_x
        + np.log(
            -np.expm1(delta)
        )
    )


def python_log_normal_interval(lower, upper):
    if np.isneginf(lower):
        return float(
            log_ndtr(
                upper
            )
        )

    if np.isposinf(upper):
        return float(
            log_ndtr(
                -lower
            )
        )

    if upper <= 0.0:
        return python_logdiffexp(
            float(
                log_ndtr(
                    upper
                )
            ),
            float(
                log_ndtr(
                    lower
                )
            ),
        )

    if lower >= 0.0:
        return python_logdiffexp(
            float(
                log_ndtr(
                    -lower
                )
            ),
            float(
                log_ndtr(
                    -upper
                )
            ),
        )

    probability = (
        ndtr(
            upper
        )
        - ndtr(
            lower
        )
    )

    if probability <= 0.0:
        raise RuntimeError(
            f"Central Normal interval probability became nonpositive: "
            f"lower={lower}, upper={upper}"
        )

    return float(
        np.log(
            probability
        )
    )


def python_deviance(m, p, y):
    value = 0.0

    if y > 0:
        value += (
            y
            * np.log(
                y
                / (
                    m * p
                )
            )
        )

    if y < m:
        value += (
            (
                m - y
            )
            * np.log(
                (
                    m - y
                )
                / (
                    m
                    * (
                        1.0 - p
                    )
                )
            )
        )

    return max(
        2.0 * value,
        0.0,
    )


def python_signed_root(m, p, y):
    difference = (
        y
        - m * p
    )

    if difference == 0.0:
        return 0.0

    return (
        np.sign(
            difference
        )
        * np.sqrt(
            python_deviance(
                m,
                p,
                y,
            )
        )
    )


def python_bounds(m, p, y):
    z = python_signed_root(
        m,
        p,
        y,
    )

    lower = -np.inf
    upper = np.inf

    if y > 0:
        lower = 0.5 * (
            python_signed_root(
                m,
                p,
                y - 1,
            )
            + z
        )

    if y < m:
        upper = 0.5 * (
            z
            + python_signed_root(
                m,
                p,
                y + 1,
            )
        )

    return (
        lower,
        upper,
    )


def python_log_binomial(m, p, y):
    return (
        gammaln(
            m + 1
        )
        - gammaln(
            y + 1
        )
        - gammaln(
            m - y + 1
        )
        + y
        * np.log(
            p
        )
        + (
            m - y
        )
        * np.log1p(
            -p
        )
    )


def python_log_saturated(m, y):
    value = (
        gammaln(
            m + 1
        )
        - gammaln(
            y + 1
        )
        - gammaln(
            m - y + 1
        )
    )

    if y > 0:
        value += (
            y
            * np.log(
                y / m
            )
        )

    if y < m:
        value += (
            (
                m - y
            )
            * np.log(
                (
                    m - y
                )
                / m
            )
        )

    return value


def python_carriers(m, p, theta, y):
    lower, upper = python_bounds(
        m,
        p,
        y,
    )

    source = (
        python_log_binomial(
            m,
            p,
            y,
        )
        - python_log_normal_interval(
            lower,
            upper,
        )
    )

    root_theta = np.sqrt(
        theta
    )

    target = (
        theta
        * python_log_binomial(
            m,
            p,
            y,
        )
        + (
            1.0 - theta
        )
        * python_log_saturated(
            m,
            y,
        )
        - python_log_normal_interval(
            root_theta
            * lower,
            root_theta
            * upper,
        )
    )

    return (
        source,
        target,
    )


rng = np.random.default_rng(
    1
)

rows = []

for test in range(
    5000
):
    m = int(
        rng.integers(
            2,
            5001,
        )
    )

    if rng.random() < 0.5:
        exponent = rng.uniform(
            -12.0,
            -0.30103,
        )

        p = 10.0 ** exponent

        if rng.random() < 0.5:
            p = 1.0 - p

    else:
        p = rng.uniform(
            0.001,
            0.999,
        )

    theta = np.exp(
        rng.uniform(
            np.log(
                1e-4
            ),
            np.log(
                0.999
            ),
        )
    )

    candidate_y = {
        0,
        1,
        2,
        m - 2,
        m - 1,
        m,
        int(
            np.clip(
                round(
                    m * p
                ),
                0,
                m,
            )
        ),
        int(
            rng.integers(
                0,
                m + 1,
            )
        ),
    }

    y = int(
        rng.choice(
            list(
                candidate_y
            )
        )
    )

    cpp = fast_db_stage_a.stage_a_values(
        m,
        p,
        theta,
        y,
    )

    py_D = python_deviance(
        m,
        p,
        y,
    )

    py_z = python_signed_root(
        m,
        p,
        y,
    )

    py_cS, py_cT = python_carriers(
        m,
        p,
        theta,
        y,
    )

    rows.append(
        (
            abs(
                cpp[
                    "deviance"
                ]
                - py_D
            ),
            abs(
                cpp[
                    "signed_root_deviance"
                ]
                - py_z
            ),
            abs(
                cpp[
                    "source_carrier"
                ]
                - py_cS
            ),
            abs(
                cpp[
                    "target_carrier"
                ]
                - py_cT
            ),
        )
    )


error = np.asarray(
    rows,
    dtype=float,
)

print("n tests:", len(error))

print(
    "max |D cpp-python|:",
    np.max(
        error[
            :,
            0
        ]
    ),
)

print(
    "max |z cpp-python|:",
    np.max(
        error[
            :,
            1
        ]
    ),
)

print(
    "max |cS cpp-python|:",
    np.max(
        error[
            :,
            2
        ]
    ),
)

print(
    "max |cT cpp-python|:",
    np.max(
        error[
            :,
            3
        ]
    ),
)

print(
    "q99 |cS error|:",
    np.quantile(
        error[
            :,
            2
        ],
        0.99,
    ),
)

print(
    "q99 |cT error|:",
    np.quantile(
        error[
            :,
            3
        ],
        0.99,
    ),
)


n tests: 5000
max |D cpp-python|: 0.0
max |z cpp-python|: 0.0
max |cS cpp-python|: 5.820766091346741e-11
max |cT cpp-python|: 2.9103830456733704e-11
q99 |cS error|: 2.9103830456733704e-11
q99 |cT error|: 7.773000021416041e-12


In [8]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_b.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)

running build_ext
building 'fast_db_stage_b' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tpfast_db_

In [9]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(0, str(CPP_DIR))

import fast_db_stage_b

print("C++ version:", fast_db_stage_b.cpp_version())

print(
    fast_db_stage_b.fast_envelope_setup(
        m=100,
        p=0.2,
        theta=0.1,
    )
)


C++ version: C++17
{'m': 100, 'p': 0.2, 'theta': 0.1, 'target_argmax': 100, 'U_target': 2.3220167990071445, 'target_candidate_count': 6, 'source_left_ternary_argmin': 50, 'source_right_ternary_argmin': 78, 'source_argmin': 78, 'L_source': -0.14942218206141433, 'source_query_count': 32, 'M_separable': 2.471438981068559}


In [10]:
exec(
    (CPP_DIR / "audit_fast_db_stage_b.py").read_text(
        encoding="utf-8"
    )
)

stage_b_audit = audit_fast_db_stage_b(
    fast_db_stage_b,
    n_random=3000,
    random_seed=1,
    terminal_scan_width=8,
)

[1,000/4,760] source_fail=0 | target_fail=0 | M_fail=0
[2,000/4,760] source_fail=0 | target_fail=0 | M_fail=0
[3,000/4,760] source_fail=0 | target_fail=0 | M_fail=0
[4,000/4,760] source_fail=0 | target_fail=0 | M_fail=0

=== STAGE-B COMPILED ENVELOPE AUDIT ===
C++: C++17
cases: 4760
source failures: 0
target failures: 0
separable M failures: 0
max |L_source fast-full|: 0.0
max |U_target fast-full|: 0.0
max |M_sep fast-full|: 0.0
median source queries: 55.0
q95 source queries: 68.0
max source queries: 71
elapsed seconds: 10.879054800025187


In [11]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_c.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)


running build_ext
building 'fast_db_stage_c' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tpfast_db_

In [12]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(0, str(CPP_DIR))

import fast_db_stage_c

print(fast_db_stage_c.cpp_version())


C++17


In [13]:
draws = fast_db_stage_c.double_binomial(
    m=100,
    p=0.2,
    theta=0.5,
    n=10000,
    seed=1,
)

In [14]:
exec(
    (CPP_DIR / "audit_fast_db_stage_c.py").read_text(
        encoding="utf-8"
    )
)

stage_c_audit = audit_fast_db_stage_c(
    fast_db_stage_c,
    n_draws=200000,
    random_seed=1,
    verbose=1,
)

display(stage_c_audit)

[01/17] m= 10 p=0.2 theta=0.1 | mean_z=-1.774 | var_ratio=0.9990 | TV=0.00301 | CDF=0.00253 | acc=0.6183 | max_logA=+0.000e+00 | 247,040 draws/s
[02/17] m= 10 p=0.5 theta=0.5 | mean_z=+2.003 | var_ratio=0.9974 | TV=0.00331 | CDF=0.00211 | acc=0.5312 | max_logA=-4.284e-02 | 194,965 draws/s
[03/17] m= 20 p=0.05 theta=0.2 | mean_z=-1.163 | var_ratio=0.9974 | TV=0.00212 | CDF=0.00175 | acc=0.1834 | max_logA=-6.786e-01 | 44,226 draws/s
[04/17] m= 20 p=0.5 theta=0.8 | mean_z=+1.186 | var_ratio=0.9965 | TV=0.00345 | CDF=0.00173 | acc=0.5124 | max_logA=-1.452e-01 | 102,372 draws/s
[05/17] m= 50 p=0.1 theta=0.25 | mean_z=-0.168 | var_ratio=0.9998 | TV=0.00377 | CDF=0.00111 | acc=0.1908 | max_logA=-1.350e+00 | 25,222 draws/s
[06/17] m= 50 p=0.5 theta=0.5 | mean_z=-0.165 | var_ratio=1.0006 | TV=0.00446 | CDF=0.00135 | acc=0.3381 | max_logA=-8.241e-01 | 63,009 draws/s
[07/17] m=100 p=0.01 theta=0.1 | mean_z=+0.601 | var_ratio=1.0001 | TV=0.00360 | CDF=0.00218 | acc=0.0510 | max_logA=-2.430e+00 | 1

,m,p,theta,n_draws,exact_mean,sample_mean,mean_z,exact_variance,sample_variance,variance_ratio,empirical_TV,empirical_CDF_sup,max_abs_bin_z_expected20,acceptance,proposals,maximum_log_accept,source_query_count,elapsed_seconds,draws_per_second
0,10,0.20,0.10,200000,3.367178,3.354265,-1.774461,10.591666,10.580661,0.998961,0.003010,0.002530,1.705301,0.618301,323467,0.000000,11,0.809586,2.470398e+05
1,10,0.50,0.50,200000,5.000000,5.010100,2.002860,5.085945,5.072628,0.997382,0.003314,0.002108,1.881849,0.531206,376502,-0.042841,11,1.025825,1.949650e+05
2,20,0.05,0.20,200000,1.677537,1.672015,-1.163428,4.504917,4.493411,0.997446,0.002121,0.001750,1.463080,0.183354,1090789,-0.678621,18,4.522197,4.422629e+04
3,20,0.50,0.80,200000,10.000000,10.006635,1.186419,6.255120,6.233381,0.996525,0.003452,0.001734,1.819036,0.512395,390324,-0.145227,19,1.953663,1.023718e+05
4,50,0.10,0.25,200000,5.072472,5.070935,-0.168458,16.641221,16.638443,0.999833,0.003767,0.001108,1.945814,0.190810,1048165,-1.350029,25,7.929519,2.522221e+04
5,50,0.50,0.50,200000,25.000000,24.998150,-0.165413,25.016946,25.031447,1.000580,0.004461,0.001345,1.819549,0.338082,591573,-0.824117,25,3.174152,6.300896e+04
6,100,0.01,0.10,200000,2.543218,2.547745,0.600734,11.357592,11.358890,1.000114,0.003600,0.002179,2.008939,0.051006,3921098,-2.430488,33,12.183680,1.641540e+04
7,100,0.20,0.50,200000,19.992007,19.998580,0.519419,32.022762,32.011388,0.999645,0.004574,0.001623,2.674471,0.224103,892447,-0.636460,32,3.354697,5.961791e+04
8,100,0.50,0.80,200000,50.000000,49.997185,-0.225197,31.250695,31.175247,0.997586,0.003764,0.001090,2.440118,0.412265,485125,-0.862383,33,3.256709,6.141168e+04
9,200,0.05,0.25,200000,9.897041,9.902550,0.402075,37.545087,37.537093,0.999787,0.004517,0.000966,2.160901,0.085333,2343764,-1.948100,37,10.224691,1.956049e+04


In [15]:
exec(
    (CPP_DIR / "benchmark_fast_db_stage_c.py").read_text(
        encoding="utf-8"
    )
)

stage_c_benchmark = benchmark_fast_db_stage_c(
    fast_db_stage_c,
    n_repeats=7,
    random_seed=1,
)

display(stage_c_benchmark)


m=  20 p=0.2 theta=0.1 N=    1 | C++=6.68e-05s | PMF=0.0005447s | speedup=8.154x
m=  20 p=0.2 theta=0.1 N=   10 | C++=0.0001044s | PMF=0.0005551s | speedup=5.317x
m=  20 p=0.2 theta=0.1 N=  100 | C++=0.0006767s | PMF=0.0005418s | speedup=0.801x
m=  20 p=0.2 theta=0.1 N= 1000 | C++=0.0074615s | PMF=0.0013244s | speedup=0.177x
m=  20 p=0.2 theta=0.1 N=20000 | C++=0.133942s | PMF=0.0019724s | speedup=0.015x
m= 100 p=0.2 theta=0.1 N=    1 | C++=8.9e-05s | PMF=0.0005631s | speedup=6.327x
m= 100 p=0.2 theta=0.1 N=   10 | C++=0.0002167s | PMF=0.0005501s | speedup=2.539x
m= 100 p=0.2 theta=0.1 N=  100 | C++=0.0015918s | PMF=0.0006016s | speedup=0.378x
m= 100 p=0.2 theta=0.1 N= 1000 | C++=0.0140871s | PMF=0.000894s | speedup=0.063x
m= 100 p=0.2 theta=0.1 N=20000 | C++=0.290671s | PMF=0.0026198s | speedup=0.009x
m= 100 p=0.2 theta=0.5 N=    1 | C++=6.73e-05s | PMF=0.0006092s | speedup=9.052x
m= 100 p=0.2 theta=0.5 N=   10 | C++=0.0002128s | PMF=0.0005487s | speedup=2.578x
m= 100 p=0.2 theta=0.5 

,m,p,theta,n_draws,cpp_fast_total_seconds,pmf_total_seconds,pmf_setup_seconds,pmf_draw_only_seconds,cpp_over_pmf_total,speedup_cpp_vs_pmf_total
0,20,0.20,0.1,1,0.000067,0.000545,0.000519,0.000025,0.122636,8.154187
1,20,0.20,0.1,10,0.000104,0.000555,0.000532,0.000024,0.188074,5.317051
2,20,0.20,0.1,100,0.000677,0.000542,0.000509,0.000030,1.248985,0.800650
3,20,0.20,0.1,1000,0.007462,0.001324,0.000785,0.000119,5.633872,0.177498
4,20,0.20,0.1,20000,0.133942,0.001972,0.000729,0.001195,67.908234,0.014726
5,100,0.20,0.1,1,0.000089,0.000563,0.000539,0.000023,0.158054,6.326968
6,100,0.20,0.1,10,0.000217,0.000550,0.000526,0.000023,0.393928,2.538532
7,100,0.20,0.1,100,0.001592,0.000602,0.000566,0.000034,2.645944,0.377937
8,100,0.20,0.1,1000,0.014087,0.000894,0.000755,0.000134,15.757383,0.063462
9,100,0.20,0.1,20000,0.290671,0.002620,0.000779,0.001839,110.951521,0.009013


In [18]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_e.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)

running build_ext
building 'fast_db_stage_e' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tpfast_db_

In [19]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(0, str(CPP_DIR))

import fast_db_stage_e

print(fast_db_stage_e.cpp_version())

C++17


In [20]:
exec(
    (CPP_DIR / "benchmark_large_changing_parameter_simulation_timeout.py").read_text(
        encoding="utf-8"
    )
)

big_db_benchmark = benchmark_large_changing_parameter_simulation_timeout(
    fast_db_stage_e,
    n_repeats=3,

    method_timeout_seconds=120.0,
    diagnostic_timeout_seconds=120.0,

    random_seed=1,
    verbose=1,
)

display(big_db_benchmark["results"])
display(big_db_benchmark["fast_diagnostics"])
display(big_db_benchmark["workload_parameters"])


[01/09] small_m_100k_x1 | params=100,000 | draws=100,000 | median m=32 | max m=100 | support=4,009,459 | FAST=7.2489s | PMF=4.2344s | speedup=0.58x | winner=PMF
[02/09] moderate_m_50k_x1 | params=50,000 | draws=50,000 | median m=141 | max m=1,000 | support=12,586,796 | FAST=3.7170s | PMF=4.6793s | speedup=1.26x | winner=FAST
[03/09] large_m_20k_x1 | params=20,000 | draws=20,000 | median m=692 | max m=4,999 | support=24,675,546 | FAST=3.1557s | PMF=10.2369s | speedup=3.24x | winner=FAST
[04/09] very_large_m_5k_x1 | params=5,000 | draws=5,000 | median m=4,616 | max m=19,989 | support=31,861,202 | FAST=1.5849s | PMF=10.6624s | speedup=6.73x | winner=FAST
[05/09] huge_m_1k_x1 | params=1,000 | draws=1,000 | median m=23,580 | max m=99,462 | support=32,577,720 | FAST=0.6557s | PMF=11.7271s | speedup=17.88x | winner=FAST
[06/09] moderate_m_20k_x10 | params=20,000 | draws=200,000 | median m=141 | max m=1,000 | support=5,034,949 | FAST=12.1043s | PMF=2.2957s | speedup=0.19x | winner=PMF
[07/09] 

,workload,n_parameters,draws_per_parameter,total_output_draws,m_median,m_mean,m_q95,m_max,total_support_points_pmf,fast_status,...,speedup_fast_vs_pmf,winner,fast_error,pmf_error,method_timeout_seconds,diagnostic_status,acceptance,mean_proposals_per_draw,mean_source_queries_per_parameter,maximum_log_accept
0,small_m_100k_x1,100000,1,100000,32.0,39.09459,89.00,100,4009459,OK,...,0.584146,PMF,,,120.0,OK,0.076557,13.06211,21.77151,0.000000
1,moderate_m_50k_x1,50000,1,50000,141.0,250.73592,823.00,1000,12586796,OK,...,1.258887,FAST,,,120.0,OK,0.048256,20.72278,34.66494,-0.000426
2,large_m_20k_x1,20000,1,20000,692.5,1232.77730,4082.05,4999,24675546,OK,...,3.243957,FAST,,,120.0,OK,0.029115,34.34640,49.01300,-0.659919
3,very_large_m_5k_x1,5000,1,5000,4616.0,6371.24040,17088.30,19989,31861202,OK,...,6.727638,FAST,,,120.0,OK,0.018007,55.53420,66.89980,-0.771919
4,huge_m_1k_x1,1000,1,1000,23580.5,32576.72000,84795.70,99462,32577720,OK,...,17.883862,FAST,,,120.0,OK,0.008266,120.97100,82.67700,-0.804655
5,moderate_m_20k_x10,20000,10,200000,141.0,250.74745,822.00,1000,5034949,OK,...,0.189657,PMF,,,120.0,OK,0.048828,20.48003,34.64990,0.000000
6,large_m_5k_x10,5000,10,50000,694.0,1257.08780,4137.35,4998,6290439,OK,...,0.449815,PMF,,,120.0,OK,0.027988,35.72978,49.14160,-0.566513
7,moderate_m_2k_x100,2000,100,200000,137.0,250.23950,829.00,1000,502479,OK,...,0.021776,PMF,,,120.0,OK,0.048767,20.50551,34.64700,0.000000
8,hard_overdispersion_5k_x1,5000,1,5000,710.0,1244.39740,4027.30,4995,6226987,OK,...,1.913111,FAST,,,120.0,OK,0.017543,57.00280,49.13300,-0.000814


,workload,diagnostic_status,acceptance,mean_proposals_per_draw,mean_source_queries_per_parameter,maximum_log_accept
0,small_m_100k_x1,OK,0.076557,13.06211,21.77151,0.000000
1,moderate_m_50k_x1,OK,0.048256,20.72278,34.66494,-0.000426
2,large_m_20k_x1,OK,0.029115,34.34640,49.01300,-0.659919
3,very_large_m_5k_x1,OK,0.018007,55.53420,66.89980,-0.771919
4,huge_m_1k_x1,OK,0.008266,120.97100,82.67700,-0.804655
5,moderate_m_20k_x10,OK,0.048828,20.48003,34.64990,0.000000
6,large_m_5k_x10,OK,0.027988,35.72978,49.14160,-0.566513
7,moderate_m_2k_x100,OK,0.048767,20.50551,34.64700,0.000000
8,hard_overdispersion_5k_x1,OK,0.017543,57.00280,49.13300,-0.000814


,workload,n_parameters,draws_per_parameter,total_output_draws,m_min,m_median,m_mean,m_q95,m_max,total_support_points_for_pmf,mean_support_points_per_parameter,theta_min,theta_median,theta_max,p_min,p_median,p_max
0,small_m_100k_x1,100000,1,100000,10,32.0,39.09459,89.00,100,4009459,40.09459,0.100003,0.307076,0.949996,0.000100,0.499246,0.999900
1,moderate_m_50k_x1,50000,1,50000,20,141.0,250.73592,823.00,1000,12586796,251.73592,0.100010,0.307628,0.949950,0.000100,0.499968,0.999900
2,large_m_20k_x1,20000,1,20000,100,692.5,1232.77730,4082.05,4999,24675546,1233.77730,0.100033,0.310424,0.949967,0.000100,0.507196,0.999900
3,very_large_m_5k_x1,5000,1,5000,1000,4616.0,6371.24040,17088.30,19989,31861202,6372.24040,0.100043,0.312833,0.949941,0.000100,0.502750,0.999900
4,huge_m_1k_x1,1000,1,1000,5004,23580.5,32576.72000,84795.70,99462,32577720,32577.72000,0.100029,0.300637,0.948686,0.000100,0.504716,0.999899
5,moderate_m_20k_x10,20000,10,200000,20,141.0,250.74745,822.00,1000,5034949,251.74745,0.100018,0.308830,0.949842,0.000100,0.498551,0.999900
6,large_m_5k_x10,5000,10,50000,100,694.0,1257.08780,4137.35,4998,6290439,1258.08780,0.100019,0.307955,0.948206,0.000101,0.498878,0.999900
7,moderate_m_2k_x100,2000,100,200000,20,137.0,250.23950,829.00,1000,502479,251.23950,0.100080,0.313954,0.946812,0.000101,0.479274,0.999896
8,hard_overdispersion_5k_x1,5000,1,5000,100,710.0,1244.39740,4027.30,4995,6226987,1245.39740,0.030003,0.077877,0.199900,0.000101,0.497449,0.999899


In [22]:
def benchmark_numpy_pmf_vs_cpp_fast_large(
    fast_db_stage_e,
    *,
    workloads=None,
    n_repeats=3,
    method_timeout_seconds=120.0,
    random_seed=1,
    terminal_scan_width=8,
    max_proposals_per_parameter=1000000,
    verbose=1,
):
    """
    Compare the compiled O(log m) double-binomial sampler against the
    original NumPy/SciPy normalized-PMF implementation on the SAME large
    changing-parameter workloads.

    ------------------------------------------------------------------
    IMPORTANT
    ------------------------------------------------------------------

    The parameter arrays are generated once per workload and then reused
    unchanged by both methods.

    FAST:
        fast_db_stage_e.double_binomial_many_fast(...)
        One Python -> C++ call for the full workload.

    NUMPY PMF:
        For each changing (m,p,theta) triple:
            1. build y = 0,...,m
            2. compute double-binomial log weights with SciPy/NumPy
            3. normalize with scipy.special.logsumexp
            4. build the CDF
            5. draw using np.searchsorted

    This intentionally measures the NumPy/SciPy implementation as it would
    actually be used when every parameter triple changes.  Large vector
    operations are compiled internally, but changing m means a Python-level
    loop across parameter triples.

    A wall-clock guard is checked between parameter sets for the NumPy path.
    The current parameter ranges are small enough that a single NumPy support
    calculation should not itself monopolize the process for long.

    Returns
    -------
    dict with:
        results
        workload_parameters
    """

    from time import perf_counter

    import numpy as np
    import pandas as pd

    from scipy.special import gammaln, logsumexp, xlogy

    if workloads is None:
        workloads = (
            (
                "small_m_100k_x1",
                100000,
                10,
                100,
                1,
                0.10,
                0.95,
            ),
            (
                "moderate_m_50k_x1",
                50000,
                20,
                1000,
                1,
                0.10,
                0.95,
            ),
            (
                "large_m_20k_x1",
                20000,
                100,
                5000,
                1,
                0.10,
                0.95,
            ),
            (
                "very_large_m_5k_x1",
                5000,
                1000,
                20000,
                1,
                0.10,
                0.95,
            ),
            (
                "huge_m_1k_x1",
                1000,
                5000,
                100000,
                1,
                0.10,
                0.95,
            ),
            (
                "moderate_m_20k_x10",
                20000,
                20,
                1000,
                10,
                0.10,
                0.95,
            ),
            (
                "large_m_5k_x10",
                5000,
                100,
                5000,
                10,
                0.10,
                0.95,
            ),
            (
                "moderate_m_2k_x100",
                2000,
                20,
                1000,
                100,
                0.10,
                0.95,
            ),
            (
                "hard_overdispersion_5k_x1",
                5000,
                100,
                5000,
                1,
                0.03,
                0.20,
            ),
        )

    rng = np.random.default_rng(
        int(random_seed)
    )

    # ==============================================================
    # SAME workload generators as the C++ large simulation benchmark
    # ==============================================================

    def log_uniform_integer(low, high, n):
        values = np.exp(
            rng.uniform(
                np.log(float(low)),
                np.log(float(high)),
                size=int(n),
            )
        )

        return np.clip(
            np.rint(values),
            int(low),
            int(high),
        ).astype(np.int64)

    def generate_p_values(n):
        n = int(n)

        p = np.empty(
            n,
            dtype=np.float64,
        )

        central = (
            rng.random(n)
            < 0.60
        )

        n_central = int(
            np.sum(central)
        )

        p[central] = np.clip(
            rng.beta(
                2.0,
                2.0,
                size=n_central,
            ),
            1e-6,
            1.0 - 1e-6,
        )

        tail_index = np.flatnonzero(
            ~central
        )

        if len(tail_index):
            tail = np.exp(
                rng.uniform(
                    np.log(1e-4),
                    np.log(0.10),
                    size=len(tail_index),
                )
            )

            reflect = (
                rng.random(
                    len(tail_index)
                )
                < 0.5
            )

            tail[reflect] = (
                1.0
                - tail[reflect]
            )

            p[tail_index] = tail

        return np.ascontiguousarray(
            p,
            dtype=np.float64,
        )

    def generate_theta_values(
        n,
        theta_min,
        theta_max,
    ):
        return np.ascontiguousarray(
            np.exp(
                rng.uniform(
                    np.log(float(theta_min)),
                    np.log(float(theta_max)),
                    size=int(n),
                )
            ),
            dtype=np.float64,
        )

    # ==============================================================
    # NumPy/SciPy PMF implementation
    # ==============================================================

    def numpy_double_binomial_pmf_draws(
        m_values,
        p_values,
        theta_values,
        draws_per_parameter,
        seed,
        timeout_seconds,
    ):
        local_rng = np.random.default_rng(
            int(seed)
        )

        n_parameters = len(
            m_values
        )

        output = np.empty(
            (
                n_parameters,
                int(draws_per_parameter),
            ),
            dtype=np.int64,
        )

        start_time = perf_counter()

        for index in range(
            n_parameters
        ):
            if (
                timeout_seconds > 0.0
                and (
                    perf_counter()
                    - start_time
                )
                >= timeout_seconds
            ):
                raise TimeoutError(
                    "NUMPY_TIMEOUT: wall-clock limit exceeded."
                )

            m = int(
                m_values[index]
            )

            p = float(
                p_values[index]
            )

            theta = float(
                theta_values[index]
            )

            y = np.arange(
                m + 1,
                dtype=np.float64,
            )

            log_choose = (
                gammaln(
                    m + 1.0
                )
                - gammaln(
                    y + 1.0
                )
                - gammaln(
                    m - y + 1.0
                )
            )

            log_binomial = (
                log_choose
                + xlogy(
                    y,
                    p
                )
                + xlogy(
                    m - y,
                    1.0 - p
                )
            )

            q = (
                y
                / float(m)
            )

            log_saturated = (
                log_choose
                + xlogy(
                    y,
                    q
                )
                + xlogy(
                    m - y,
                    1.0 - q
                )
            )

            log_weight = (
                theta
                * log_binomial
                + (
                    1.0 - theta
                )
                * log_saturated
            )

            probability = np.exp(
                log_weight
                - logsumexp(
                    log_weight
                )
            )

            cdf = np.cumsum(
                probability
            )

            cdf[-1] = 1.0

            u = local_rng.random(
                int(
                    draws_per_parameter
                )
            )

            output[
                index,
                :
            ] = np.searchsorted(
                cdf,
                u,
                side="right",
            )

        return output

    # ==============================================================
    # Warm-up
    # ==============================================================

    warm_m = np.asarray(
        [20, 100, 500],
        dtype=np.int64,
    )

    warm_p = np.asarray(
        [0.2, 0.01, 0.5],
        dtype=np.float64,
    )

    warm_theta = np.asarray(
        [0.5, 0.2, 0.8],
        dtype=np.float64,
    )

    fast_db_stage_e.double_binomial_many_fast(
        warm_m,
        warm_p,
        warm_theta,
        1,
        123,
        int(terminal_scan_width),
        int(max_proposals_per_parameter),
        5.0,
    )

    numpy_double_binomial_pmf_draws(
        warm_m,
        warm_p,
        warm_theta,
        1,
        123,
        5.0,
    )

    # ==============================================================
    # Benchmark
    # ==============================================================

    rows = []
    parameter_rows = []

    for workload_number, workload in enumerate(
        workloads,
        start=1,
    ):
        (
            name,
            n_parameters,
            m_min,
            m_max,
            draws_per_parameter,
            theta_min,
            theta_max,
        ) = workload

        n_parameters = int(
            n_parameters
        )

        draws_per_parameter = int(
            draws_per_parameter
        )

        m_values = np.ascontiguousarray(
            log_uniform_integer(
                m_min,
                m_max,
                n_parameters,
            ),
            dtype=np.int64,
        )

        p_values = generate_p_values(
            n_parameters
        )

        theta_values = generate_theta_values(
            n_parameters,
            theta_min,
            theta_max,
        )

        total_draws = (
            n_parameters
            * draws_per_parameter
        )

        total_support_points = int(
            np.sum(
                m_values
                + 1
            )
        )

        parameter_rows.append(
            {
                "workload":
                    name,
                "n_parameters":
                    n_parameters,
                "draws_per_parameter":
                    draws_per_parameter,
                "total_draws":
                    total_draws,
                "m_min":
                    int(
                        np.min(m_values)
                    ),
                "m_median":
                    float(
                        np.median(m_values)
                    ),
                "m_mean":
                    float(
                        np.mean(m_values)
                    ),
                "m_q95":
                    float(
                        np.quantile(
                            m_values,
                            0.95,
                        )
                    ),
                "m_max":
                    int(
                        np.max(m_values)
                    ),
                "total_support_points":
                    total_support_points,
                "mean_support_points":
                    float(
                        total_support_points
                        / n_parameters
                    ),
            }
        )

        cpp_times = []
        numpy_times = []

        cpp_status = "OK"
        numpy_status = "OK"

        cpp_error = ""
        numpy_error = ""

        for repeat in range(
            int(n_repeats)
        ):
            seed = int(
                random_seed
                + 1_000_003
                * workload_number
                + 10_007
                * repeat
            )

            # Alternate timing order.
            order = (
                ("cpp", "numpy")
                if repeat % 2 == 0
                else ("numpy", "cpp")
            )

            for method in order:
                if (
                    method == "cpp"
                    and cpp_status == "OK"
                ):
                    start = perf_counter()

                    try:
                        cpp_draws = (
                            fast_db_stage_e
                            .double_binomial_many_fast(
                                m_values,
                                p_values,
                                theta_values,
                                draws_per_parameter,
                                seed,
                                int(
                                    terminal_scan_width
                                ),
                                int(
                                    max_proposals_per_parameter
                                ),
                                float(
                                    method_timeout_seconds
                                ),
                            )
                        )

                        elapsed = (
                            perf_counter()
                            - start
                        )

                        _ = int(
                            np.asarray(
                                cpp_draws
                            )[
                                : min(
                                    100,
                                    n_parameters
                                )
                            ].sum()
                        )

                        cpp_times.append(
                            elapsed
                        )

                    except RuntimeError as error:
                        if "FAST_TIMEOUT:" not in str(
                            error
                        ):
                            raise

                        cpp_status = "TIMEOUT"
                        cpp_error = str(
                            error
                        )

                if (
                    method == "numpy"
                    and numpy_status == "OK"
                ):
                    start = perf_counter()

                    try:
                        numpy_draws = (
                            numpy_double_binomial_pmf_draws(
                                m_values,
                                p_values,
                                theta_values,
                                draws_per_parameter,
                                seed,
                                float(
                                    method_timeout_seconds
                                ),
                            )
                        )

                        elapsed = (
                            perf_counter()
                            - start
                        )

                        _ = int(
                            numpy_draws[
                                : min(
                                    100,
                                    n_parameters
                                )
                            ].sum()
                        )

                        numpy_times.append(
                            elapsed
                        )

                    except TimeoutError as error:
                        numpy_status = "TIMEOUT"
                        numpy_error = str(
                            error
                        )

        cpp_median = (
            float(
                np.median(
                    cpp_times
                )
            )
            if cpp_times
            else np.nan
        )

        numpy_median = (
            float(
                np.median(
                    numpy_times
                )
            )
            if numpy_times
            else np.nan
        )

        speedup = (
            float(
                numpy_median
                / cpp_median
            )
            if (
                np.isfinite(
                    cpp_median
                )
                and np.isfinite(
                    numpy_median
                )
                and cpp_median > 0.0
            )
            else np.nan
        )

        if (
            cpp_status == "TIMEOUT"
            and numpy_status == "OK"
        ):
            winner = "NUMPY PMF"

        elif (
            numpy_status == "TIMEOUT"
            and cpp_status == "OK"
        ):
            winner = "CPP FAST"

        elif (
            cpp_status == "TIMEOUT"
            and numpy_status == "TIMEOUT"
        ):
            winner = "BOTH TIMEOUT"

        elif speedup > 1.0:
            winner = "CPP FAST"

        else:
            winner = "NUMPY PMF"

        rows.append(
            {
                "workload":
                    name,

                "n_parameters":
                    n_parameters,

                "draws_per_parameter":
                    draws_per_parameter,

                "total_draws":
                    total_draws,

                "m_median":
                    float(
                        np.median(
                            m_values
                        )
                    ),

                "m_mean":
                    float(
                        np.mean(
                            m_values
                        )
                    ),

                "m_q95":
                    float(
                        np.quantile(
                            m_values,
                            0.95,
                        )
                    ),

                "m_max":
                    int(
                        np.max(
                            m_values
                        )
                    ),

                "total_support_points_numpy":
                    total_support_points,

                "cpp_status":
                    cpp_status,

                "numpy_status":
                    numpy_status,

                "cpp_fast_median_seconds":
                    cpp_median,

                "numpy_pmf_median_seconds":
                    numpy_median,

                "speedup_cpp_fast_vs_numpy_pmf":
                    speedup,

                "winner":
                    winner,

                "cpp_parameters_per_second":
                    (
                        float(
                            n_parameters
                            / cpp_median
                        )
                        if np.isfinite(
                            cpp_median
                        )
                        else np.nan
                    ),

                "numpy_parameters_per_second":
                    (
                        float(
                            n_parameters
                            / numpy_median
                        )
                        if np.isfinite(
                            numpy_median
                        )
                        else np.nan
                    ),

                "cpp_draws_per_second":
                    (
                        float(
                            total_draws
                            / cpp_median
                        )
                        if np.isfinite(
                            cpp_median
                        )
                        else np.nan
                    ),

                "numpy_draws_per_second":
                    (
                        float(
                            total_draws
                            / numpy_median
                        )
                        if np.isfinite(
                            numpy_median
                        )
                        else np.nan
                    ),

                "cpp_error":
                    cpp_error,

                "numpy_error":
                    numpy_error,

                "method_timeout_seconds":
                    float(
                        method_timeout_seconds
                    ),
            }
        )

        if verbose:
            cpp_text = (
                f"{cpp_median:.4f}s"
                if np.isfinite(
                    cpp_median
                )
                else cpp_status
            )

            numpy_text = (
                f"{numpy_median:.4f}s"
                if np.isfinite(
                    numpy_median
                )
                else numpy_status
            )

            speedup_text = (
                f"{speedup:.2f}x"
                if np.isfinite(
                    speedup
                )
                else "NA"
            )

            print(
                f"[{workload_number:02d}/{len(workloads):02d}] "
                f"{name} | "
                f"params={n_parameters:,} | "
                f"draws={total_draws:,} | "
                f"median m={np.median(m_values):,.0f} | "
                f"max m={np.max(m_values):,} | "
                f"support={total_support_points:,} | "
                f"CPP={cpp_text} | "
                f"NumPy={numpy_text} | "
                f"speedup={speedup_text} | "
                f"winner={winner}"
            )

    results = pd.DataFrame(
        rows
    )

    workload_parameters = pd.DataFrame(
        parameter_rows
    )

    print(
        "\n=== CPP FAST VS NUMPY/SCIPY PMF: LARGE CHANGING-PARAMETER WORKLOADS ==="
    )

    print(
        results.to_string(
            index=False
        )
    )

    return {
        "results":
            results,

        "workload_parameters":
            workload_parameters,
    }

# exec(
#     (CPP_DIR / "benchmark_numpy_pmf_vs_cpp_fast_large.py").read_text(
#         encoding="utf-8"
#     )
# )

numpy_vs_cpp = benchmark_numpy_pmf_vs_cpp_fast_large(
    fast_db_stage_e,

    n_repeats=3,
    method_timeout_seconds=120.0,

    random_seed=1,
    verbose=1,
)

display(numpy_vs_cpp["results"])
display(numpy_vs_cpp["workload_parameters"])


[01/09] small_m_100k_x1 | params=100,000 | draws=100,000 | median m=32 | max m=100 | support=4,009,459 | CPP=9.6907s | NumPy=64.2876s | speedup=6.63x | winner=CPP FAST
[02/09] moderate_m_50k_x1 | params=50,000 | draws=50,000 | median m=141 | max m=1,000 | support=12,586,796 | CPP=6.4166s | NumPy=32.6973s | speedup=5.10x | winner=CPP FAST


KeyboardInterrupt: 

In [23]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_f.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)


running build_ext
building 'fast_db_stage_f' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tpfast_db_

In [24]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(0, str(CPP_DIR))

import fast_db_stage_f

print(fast_db_stage_f.cpp_version())

C++17


In [30]:
exec(
    (CPP_DIR / "benchmark_optimized_double_binomial_large.py").read_text(
        encoding="utf-8"
    )
)

optimized_db_benchmark = benchmark_optimized_double_binomial_large(
    fast_db_stage_f,

    n_repeats=3,
    method_timeout_seconds=120.0,

    validation_cases=1000,
    random_seed=1,

    include_numpy=True,
    verbose=1,
)

display(optimized_db_benchmark["audit"])
display(optimized_db_benchmark["results"])
display(optimized_db_benchmark["workload_parameters"])
display(optimized_db_benchmark["fast_diagnostic"])




=== OPTIMIZED IMPLEMENTATION AUDIT ===
 validation_cases  max_abs_pmf_logweight_error  q99_abs_pmf_logweight_error  max_abs_U_target_error  max_abs_L_source_error  max_abs_M_error
             1000                 1.746230e-10                 2.183242e-11            1.711697e-11            6.708945e-11     6.387779e-11
[01/09] small_m_100k_x1 | params=100,000 | draws=100,000 | med m=32 | max m=100 | FAST=1.1838s | C++ PMF=0.0594s | NumPy=23.2741s | FAST/CPP-PMF speedup=0.05x
[02/09] moderate_m_50k_x1 | params=50,000 | draws=50,000 | med m=140 | max m=1,000 | FAST=1.5760s | C++ PMF=0.2443s | NumPy=16.6883s | FAST/CPP-PMF speedup=0.16x
[03/09] large_m_20k_x1 | params=20,000 | draws=20,000 | med m=695 | max m=4,999 | FAST=0.8485s | C++ PMF=0.5112s | NumPy=11.5643s | FAST/CPP-PMF speedup=0.60x
[04/09] very_large_m_5k_x1 | params=5,000 | draws=5,000 | med m=4,481 | max m=19,997 | FAST=0.3139s | C++ PMF=0.6016s | NumPy=5.7371s | FAST/CPP-PMF speedup=1.92x
[05/09] huge_m_1k_x1 | params=1,000

,validation_cases,max_abs_pmf_logweight_error,q99_abs_pmf_logweight_error,max_abs_U_target_error,max_abs_L_source_error,max_abs_M_error
0,1000,1.746230e-10,2.183242e-11,1.711697e-11,6.708945e-11,6.387779e-11


,workload,n_parameters,draws_per_parameter,total_draws,m_median,m_q95,m_max,total_support_points,cpp_fast_status,cpp_pmf_status,numpy_status,cpp_fast_seconds,cpp_pmf_seconds,numpy_pmf_seconds,speedup_fast_vs_optimized_cpp_pmf,speedup_fast_vs_numpy_pmf
0,small_m_100k_x1,100000,1,100000,32.0,89.00,100,4012105,OK,OK,OK,1.183809,0.059379,23.274126,0.050160,19.660371
1,moderate_m_50k_x1,50000,1,50000,140.0,823.00,1000,12572237,OK,OK,OK,1.575971,0.244284,16.688285,0.155005,10.589211
2,large_m_20k_x1,20000,1,20000,695.0,4113.00,4999,24778276,OK,OK,OK,0.848518,0.511181,11.564271,0.602439,13.628786
3,very_large_m_5k_x1,5000,1,5000,4481.0,17156.00,19997,31655742,OK,OK,OK,0.313863,0.601610,5.737067,1.916792,18.278876
4,huge_m_1k_x1,1000,1,1000,22357.0,87136.50,99906,31763348,OK,OK,OK,0.476462,2.547875,5.282660,5.347490,11.087268
5,moderate_m_20k_x10,20000,10,200000,141.0,822.00,1000,5038693,OK,OK,OK,2.709801,0.066513,5.349588,0.024545,1.974163
6,large_m_5k_x10,5000,10,50000,719.5,4125.25,4998,6274068,OK,OK,OK,1.319491,0.097865,2.044653,0.074169,1.549577
7,moderate_m_2k_x100,2000,100,200000,143.5,827.00,998,510501,OK,OK,OK,2.480073,0.013715,0.525997,0.005530,0.212089
8,hard_overdispersion_5k_x1,5000,1,5000,705.5,4100.15,4999,6219754,OK,OK,OK,0.245763,0.074505,1.873304,0.303159,7.622405


,workload,n_parameters,draws_per_parameter,total_draws,m_median,m_mean,m_q95,m_max,total_support_points
0,small_m_100k_x1,100000,1,100000,32.0,39.12105,89.00,100,4012105
1,moderate_m_50k_x1,50000,1,50000,140.0,250.44474,823.00,1000,12572237
2,large_m_20k_x1,20000,1,20000,695.0,1237.91380,4113.00,4999,24778276
3,very_large_m_5k_x1,5000,1,5000,4481.0,6330.14840,17156.00,19997,31655742
4,huge_m_1k_x1,1000,1,1000,22357.0,31762.34800,87136.50,99906,31763348
5,moderate_m_20k_x10,20000,10,200000,141.0,250.93465,822.00,1000,5038693
6,large_m_5k_x10,5000,10,50000,719.5,1253.81360,4125.25,4998,6274068
7,moderate_m_2k_x100,2000,100,200000,143.5,254.25050,827.00,998,510501
8,hard_overdispersion_5k_x1,5000,1,5000,705.5,1242.95080,4100.15,4999,6219754


,acceptance,mean_source_queries_per_parameter,maximum_log_accept
0,0.042849,44.6,-0.262487


In [26]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_g.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)

running build_ext
building 'fast_db_stage_g' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tpfast_db_

In [27]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(0, str(CPP_DIR))

import fast_db_stage_g

print(fast_db_stage_g.cpp_version())

C++17


In [28]:
exec(
    (CPP_DIR / "test_target_radius_theory.py").read_text(
        encoding="utf-8"
    )
)

target_radius_test = test_target_radius_theory(
    fast_db_stage_g,

    phase_timeout_seconds=120.0,

    random_seed=1,
    n_random=20000,

    stop_on_first_failure=False,
    verbose=1,
)



[1/5] m_2_to_5_dense: 50,616 full-support cases
  processed: 50,616/50,616
  timeout: False
  m>=6 failures: 0
  max required radius m>=6: 0
  max radius1 gap m>=6: 0.0
  min edge advantage over interior m>=6: inf

[2/5] m_6_to_50_dense: 569,430 full-support cases
  processed: 569,430/569,430
  timeout: False
  m>=6 failures: 0
  max required radius m>=6: 1
  max radius1 gap m>=6: 0.0
  min edge advantage over interior m>=6: 0.0111314816320629

[3/5] m_60_to_1000_dense: 126,540 full-support cases
  processed: 126,540/126,540
  timeout: False
  m>=6 failures: 0
  max required radius m>=6: 1
  max radius1 gap m>=6: 0.0
  min edge advantage over interior m>=6: 0.10591125980278004

[4/5] m_2000_to_20000_grid: 9,360 full-support cases
  processed: 9,360/9,360
  timeout: False
  m>=6 failures: 0
  max required radius m>=6: 1
  max radius1 gap m>=6: 0.0
  min edge advantage over interior m>=6: 0.18062930129176635

[5/5] m_6_to_5000_random: 20,000 full-support cases
  processed: 20,000/20,000

,requested_cases,processed_cases,timed_out,n_m_le_5,max_required_radius_m_le_5,n_m_ge_6,m_ge_6_radius1_failures,max_required_radius_m_ge_6,maximum_radius1_gap_m_ge_6,minimum_edge_advantage_over_interior_m_ge_6,hypothesis_passed_all_processed,phase
0,50616,50616,False,50616,2,0,0,0,0.0,inf,True,m_2_to_5_dense
1,569430,569430,False,0,0,569430,0,1,0.0,0.011131,True,m_6_to_50_dense
2,126540,126540,False,0,0,126540,0,1,0.0,0.105911,True,m_60_to_1000_dense
3,9360,9360,False,0,0,9360,0,1,0.0,0.180629,True,m_2000_to_20000_grid
4,20000,20000,False,0,0,20000,0,1,0.0,0.015082,True,m_6_to_5000_random



=== M>=6 COUNTEREXAMPLES ===
None among all processed cases.

=== CLOSEST INTERIOR CHALLENGERS ===


,phase,m,p,theta,true_argmax,true_max,radius1_argmax,radius1_max,interior_argmax,interior_max,required_radius,radius1_gap,edge_advantage_over_interior,radius1_pass
0,m_6_to_50_dense,6,0.500000,0.000001,1,6.885684,1,6.885684,2,6.874552,1,0.0,0.011131,True
1,m_6_to_50_dense,6,0.500000,0.000001,1,6.745233,1,6.745233,2,6.734101,1,0.0,0.011132,True
2,m_6_to_50_dense,6,0.500000,0.000002,1,6.604782,1,6.604782,2,6.593650,1,0.0,0.011132,True
3,m_6_to_50_dense,6,0.500000,0.000002,1,6.464331,1,6.464331,2,6.453199,1,0.0,0.011132,True
4,m_6_to_50_dense,6,0.499999,0.000001,5,6.885684,5,6.885684,4,6.874553,1,0.0,0.011132,True
5,m_6_to_50_dense,6,0.500001,0.000001,1,6.885684,1,6.885684,2,6.874553,1,0.0,0.011132,True
6,m_6_to_50_dense,6,0.499999,0.000001,5,6.745233,5,6.745233,4,6.734101,1,0.0,0.011132,True
7,m_6_to_50_dense,6,0.500001,0.000001,1,6.745233,1,6.745233,2,6.734101,1,0.0,0.011132,True
8,m_6_to_50_dense,6,0.500000,0.000003,1,6.323880,1,6.323880,2,6.312748,1,0.0,0.011132,True
9,m_6_to_50_dense,6,0.499999,0.000002,5,6.604782,5,6.604782,4,6.593650,1,0.0,0.011132,True



=== REQUIRED RADIUS HISTOGRAM ===


,phase,required_radius,n_cases
0,m_2000_to_20000_grid,0,3696
1,m_2000_to_20000_grid,1,5664
2,m_2_to_5_dense,0,24374
3,m_2_to_5_dense,1,25665
4,m_2_to_5_dense,2,577
5,m_60_to_1000_dense,0,61432
6,m_60_to_1000_dense,1,65108
7,m_6_to_5000_random,0,5004
8,m_6_to_5000_random,1,14996
9,m_6_to_50_dense,0,271407



HYPOTHESIS STATUS:
NO COUNTEREXAMPLE FOUND: every tested m>=6 case was captured by radius 1.


In [31]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "setup_stage_h.py",
        "build_ext",
        "--inplace",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)



running build_ext
building 'fast_db_stage_h' extension
"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\bin\HostX86\x64\cl.exe" /c /nologo /O2 /W3 /GL /DNDEBUG /MD -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\Lib\site-packages\pybind11\include -IC:\Users\sruddy1\ds\ds-repositories\ds-incoming-retention-risk\.venv\include -IC:\ProgramData\anaconda3\include -IC:\ProgramData\anaconda3\Include "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Tools\MSVC\14.51.36231\include" "-IC:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\VS\include" "-IC:\Program Files (x86)\Windows Kits\10\include\10.0.28000.0\ucrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\um" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\shared" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\winrt" "-IC:\Program Files (x86)\Windows Kits\10\\include\10.0.28000.0\\cppwinrt" /EHsc /Tpfast_db_

In [32]:
import sys

if str(CPP_DIR) not in sys.path:
    sys.path.insert(0, str(CPP_DIR))

import fast_db_stage_h

print(fast_db_stage_h.cpp_version())



C++17


In [35]:
from pathlib import Path

# ============================================================
# STAGE H — RUN BENCHMARK
# ============================================================

benchmark_file = Path(r"C:\Users\sruddy1\Documents\satuRnPaper\double_binomial_cpp\benchmark_stage_h.py")

if not benchmark_file.exists():
    raise FileNotFoundError(
        f"Could not find {benchmark_file.resolve()}"
    )

exec(
    benchmark_file.read_text(
        encoding="utf-8"
    )
)

Loaded: fast_db_stage_h
Compiler standard: C++17

=== STAGE H K SWEEP: 5,000 changing parameter triples, 1 draw each ===
 K  seconds  parameters_per_second  acceptance  mean_source_exact_queries  source_central_active_fraction   mean_M  maximum_log_accept status
 4 0.507513            9851.966325    0.009899                        8.0                             1.0 4.242971           -1.364407     OK
 8 0.269236           18571.053965    0.018830                       16.0                             1.0 3.692838           -1.017921     OK
16 0.266219           18781.515384    0.023933                       32.0                             1.0 3.468170           -0.879822     OK
32 0.240248           20811.819114    0.029485                       64.0                             1.0 3.367737           -0.820583     OK

Selected Stage-H bound_K: 32

=== ENVELOPE SETUP ONLY: 20,000 changing parameter triples ===
               method  seconds  parameters_per_second  mean_source_queries 

In [36]:
import gc
import time
import numpy as np
import pandas as pd

# ============================================================
# STAGE H — K=32 VS LARGER K
#
# Requires:
#     import fast_db_stage_h
#
# already compiled/imported.
# ============================================================

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

MASTER_SEED = 20260912
REPEATS = 3

K_VALUES = [
    32,
    64,
    128,
    256,
    512,
    1024,
    2048,
]

MAX_PROPOSALS_PER_PARAMETER = 0
TIMEOUT_SECONDS = 300.0


# ------------------------------------------------------------
# Workloads
#
# Larger K is only tested when 2*K <= minimum m in the
# workload. This prevents the small-support full-scan fallback
# from contaminating the comparison.
# ------------------------------------------------------------

WORKLOADS = [
    {
        "label": "m=1,000..10,000 | N=1",
        "m_low": 1_000,
        "m_high": 10_000,
        "n_parameters": 5_000,
        "draws_per_parameter": 1,
    },
    {
        "label": "m=1,000..10,000 | N=10",
        "m_low": 1_000,
        "m_high": 10_000,
        "n_parameters": 1_000,
        "draws_per_parameter": 10,
    },
    {
        "label": "m=10,000..100,000 | N=1",
        "m_low": 10_000,
        "m_high": 100_000,
        "n_parameters": 1_000,
        "draws_per_parameter": 1,
    },
]


# ------------------------------------------------------------
# Parameter generation
#
# Same distribution as the earlier Stage H benchmark:
#
#     m       uniform integer in workload range
#     logit p uniform(-8, 8)
#     theta   uniform(.05, .95)
# ------------------------------------------------------------

def logistic(x):
    return 1.0 / (1.0 + np.exp(-x))


def make_parameters(
    n_parameters,
    m_low,
    m_high,
    seed,
):
    rng = np.random.default_rng(seed)

    m_values = rng.integers(
        m_low,
        m_high + 1,
        size=n_parameters,
        dtype=np.int64,
    )

    logits = rng.uniform(
        -8.0,
        8.0,
        size=n_parameters,
    )

    p_values = logistic(logits)

    p_values = np.clip(
        p_values,
        1e-8,
        1.0 - 1e-8,
    ).astype(np.float64)

    theta_values = rng.uniform(
        0.05,
        0.95,
        size=n_parameters,
    ).astype(np.float64)

    return (
        np.ascontiguousarray(m_values),
        np.ascontiguousarray(p_values),
        np.ascontiguousarray(theta_values),
    )


# ------------------------------------------------------------
# One timed Stage H call
# ------------------------------------------------------------

def run_stage_h(
    m_values,
    p_values,
    theta_values,
    draws_per_parameter,
    bound_k,
    seed,
):
    start = time.perf_counter()

    result = (
        fast_db_stage_h
        .double_binomial_many_fast_constant_optimized_diagnostics(
            m_values,
            p_values,
            theta_values,
            draws_per_parameter=draws_per_parameter,
            seed=seed,
            bound_K=bound_k,
            max_proposals_per_parameter=(
                MAX_PROPOSALS_PER_PARAMETER
            ),
            timeout_seconds=TIMEOUT_SECONDS,
        )
    )

    elapsed = time.perf_counter() - start

    return elapsed, result


# ============================================================
# BENCHMARK
# ============================================================

all_rows = []

for workload_index, workload in enumerate(WORKLOADS):

    label = workload["label"]

    m_low = workload["m_low"]
    m_high = workload["m_high"]

    n_parameters = workload["n_parameters"]
    draws_per_parameter = workload["draws_per_parameter"]

    total_draws = (
        n_parameters
        * draws_per_parameter
    )

    print()
    print("=" * 78)
    print(label)
    print("=" * 78)

    # --------------------------------------------------------
    # Generate ONE parameter array.
    #
    # Every K gets exactly the same parameter triples.
    # --------------------------------------------------------

    m_values, p_values, theta_values = make_parameters(
        n_parameters=n_parameters,
        m_low=m_low,
        m_high=m_high,
        seed=MASTER_SEED + workload_index,
    )

    # --------------------------------------------------------
    # Avoid any K that would trigger m < 2*K for even the
    # smallest m in this workload.
    # --------------------------------------------------------

    valid_k_values = [
        k
        for k in K_VALUES
        if 2 * k <= m_low
    ]

    if 32 not in valid_k_values:
        raise RuntimeError(
            f"K=32 is not valid for workload {label}."
        )

    print("K values:", valid_k_values)

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    warm_n = min(
        64,
        n_parameters,
    )

    for bound_k in valid_k_values:

        fast_db_stage_h \
            .double_binomial_many_fast_constant_optimized(
                m_values[:warm_n],
                p_values[:warm_n],
                theta_values[:warm_n],
                draws_per_parameter=1,
                seed=900_000 + bound_k,
                bound_K=bound_k,
                max_proposals_per_parameter=(
                    MAX_PROPOSALS_PER_PARAMETER
                ),
                timeout_seconds=TIMEOUT_SECONDS,
            )

    # --------------------------------------------------------
    # Timed repetitions
    # --------------------------------------------------------

    workload_rows = []

    for bound_k in valid_k_values:

        times = []
        diagnostic_result = None

        gc.collect()

        for repeat in range(REPEATS):

            elapsed, result = run_stage_h(
                m_values=m_values,
                p_values=p_values,
                theta_values=theta_values,
                draws_per_parameter=draws_per_parameter,
                bound_k=bound_k,

                # Same seed for every K and every repeat.
                #
                # This does NOT force identical proposal paths
                # because acceptance differs, but it removes one
                # unnecessary source of variation.
                seed=(
                    700_000
                    + workload_index
                ),
            )

            times.append(
                elapsed
            )

            diagnostic_result = result

        median_seconds = float(
            np.median(
                times
            )
        )

        acceptance = float(
            diagnostic_result[
                "acceptance"
            ]
        )

        proposals = int(
            diagnostic_result[
                "proposals"
            ]
        )

        mean_M = float(
            diagnostic_result[
                "mean_M"
            ]
        )

        mean_U_target = float(
            diagnostic_result[
                "mean_U_target"
            ]
        )

        mean_L_source = float(
            diagnostic_result[
                "mean_L_source"
            ]
        )

        source_queries = float(
            diagnostic_result[
                "mean_source_exact_queries_per_setup"
            ]
        )

        central_active = float(
            diagnostic_result[
                "source_central_active_fraction"
            ]
        )

        max_log_accept = float(
            diagnostic_result[
                "maximum_log_accept"
            ]
        )

        workload_rows.append(
            {
                "workload": label,
                "K": bound_k,

                "median_seconds": (
                    median_seconds
                ),

                "min_seconds": float(
                    np.min(
                        times
                    )
                ),

                "max_seconds": float(
                    np.max(
                        times
                    )
                ),

                "parameters_per_second": (
                    n_parameters
                    / median_seconds
                ),

                "draws_per_second": (
                    total_draws
                    / median_seconds
                ),

                "acceptance": acceptance,

                "proposals_per_draw": (
                    proposals
                    / total_draws
                ),

                "mean_source_queries": (
                    source_queries
                ),

                "central_bound_active": (
                    central_active
                ),

                "mean_M": mean_M,

                "mean_U_target": (
                    mean_U_target
                ),

                "mean_L_source": (
                    mean_L_source
                ),

                "maximum_log_accept": (
                    max_log_accept
                ),
            }
        )

        print(
            f"K={bound_k:4d} | "
            f"time={median_seconds:8.4f}s | "
            f"accept={acceptance:9.6f} | "
            f"props/draw={proposals / total_draws:8.3f} | "
            f"M={mean_M:9.5f} | "
            f"source queries={source_queries:7.2f}"
        )

    # --------------------------------------------------------
    # Convert to DataFrame and compare everything to K=32
    # --------------------------------------------------------

    workload_df = pd.DataFrame(
        workload_rows
    )

    baseline = workload_df.loc[
        workload_df["K"] == 32
    ].iloc[0]

    workload_df[
        "speedup_vs_K32"
    ] = (
        baseline["median_seconds"]
        / workload_df["median_seconds"]
    )

    workload_df[
        "acceptance_ratio_vs_K32"
    ] = (
        workload_df["acceptance"]
        / baseline["acceptance"]
    )

    workload_df[
        "proposal_ratio_vs_K32"
    ] = (
        workload_df["proposals_per_draw"]
        / baseline["proposals_per_draw"]
    )

    workload_df[
        "delta_mean_M_vs_K32"
    ] = (
        workload_df["mean_M"]
        - baseline["mean_M"]
    )

    workload_df[
        "delta_L_source_vs_K32"
    ] = (
        workload_df["mean_L_source"]
        - baseline["mean_L_source"]
    )

    all_rows.extend(
        workload_df.to_dict(
            orient="records"
        )
    )

    print()
    print(
        workload_df[
            [
                "K",
                "median_seconds",
                "speedup_vs_K32",
                "acceptance",
                "acceptance_ratio_vs_K32",
                "proposals_per_draw",
                "mean_source_queries",
                "central_bound_active",
                "mean_M",
                "delta_mean_M_vs_K32",
                "delta_L_source_vs_K32",
                "maximum_log_accept",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# COMBINED RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_rows
)

print()
print()
print("=" * 100)
print("COMBINED K=32 VS LARGER-K RESULTS")
print("=" * 100)

print(
    results_df[
        [
            "workload",
            "K",
            "median_seconds",
            "speedup_vs_K32",
            "acceptance",
            "acceptance_ratio_vs_K32",
            "proposals_per_draw",
            "mean_source_queries",
            "central_bound_active",
            "mean_M",
            "delta_mean_M_vs_K32",
            "delta_L_source_vs_K32",
            "maximum_log_accept",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# BEST K PER WORKLOAD
# ============================================================

best_rows = []

for workload_name, group in results_df.groupby(
    "workload",
    sort=False,
):

    best = group.loc[
        group[
            "median_seconds"
        ].idxmin()
    ]

    best_rows.append(
        {
            "workload": workload_name,
            "best_K": int(
                best["K"]
            ),
            "best_seconds": float(
                best[
                    "median_seconds"
                ]
            ),
            "speedup_vs_K32": float(
                best[
                    "speedup_vs_K32"
                ]
            ),
            "acceptance": float(
                best[
                    "acceptance"
                ]
            ),
            "acceptance_gain_vs_K32": float(
                best[
                    "acceptance_ratio_vs_K32"
                ]
            ),
        }
    )

best_df = pd.DataFrame(
    best_rows
)

print()
print("=" * 100)
print("BEST K FOR EACH WORKLOAD")
print("=" * 100)

print(
    best_df.to_string(
        index=False
    )
)



m=1,000..10,000 | N=1
K values: [32, 64, 128, 256]
K=  32 | time=  0.2687s | accept= 0.021469 | props/draw=  46.578 | M=  3.59847 | source queries=  64.00
K=  64 | time=  0.2992s | accept= 0.022022 | props/draw=  45.410 | M=  3.54929 | source queries= 128.00
K= 128 | time=  0.3687s | accept= 0.022739 | props/draw=  43.978 | M=  3.52505 | source queries= 256.00
K= 256 | time=  0.5084s | accept= 0.023446 | props/draw=  42.650 | M=  3.51206 | source queries= 512.00

  K  median_seconds  speedup_vs_K32  acceptance  acceptance_ratio_vs_K32  proposals_per_draw  mean_source_queries  central_bound_active   mean_M  delta_mean_M_vs_K32  delta_L_source_vs_K32  maximum_log_accept
 32        0.268723        1.000000    0.021469                 1.000000             46.5778                 64.0                1.0000 3.598471             0.000000               0.000000           -0.866541
 64        0.299229        0.898052    0.022022                 1.025717             45.4100                128.0

In [40]:
import sys
import subprocess
import importlib


# ============================================================
# STAGE I — COMPILE + IMPORT
# ============================================================

CPP_DIR = Path(r"C:\Users\sruddy1\Documents\satuRnPaper\double_binomial_cpp")

subprocess.run(
    [
        sys.executable,
        "setup_stage_i.py",
        "build_ext",
        "--inplace",
        "--force",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
    check=True,
)

importlib.invalidate_caches()

import fast_db_stage_h
import fast_db_stage_i

print("Stage H:", fast_db_stage_h.cpp_version())
print("Stage I:", fast_db_stage_i.cpp_version())

print(
    "Stage I pair RNG:",
    hasattr(
        fast_db_stage_i,
        "double_binomial_many_fast_pair_optimized",
    ),
)

print(
    "Stage I pair diagnostics:",
    hasattr(
        fast_db_stage_i,
        "double_binomial_many_fast_pair_optimized_diagnostics",
    ),
)

print(
    "Stage I pair envelope:",
    hasattr(
        fast_db_stage_i,
        "fast_envelope_setup_pair_optimized",
    ),
)


Stage H: C++17
Stage I: C++17
Stage I pair RNG: True
Stage I pair diagnostics: True
Stage I pair envelope: True


In [43]:
from pathlib import Path

# ============================================================
# STAGE H K=32 VS STAGE I EXACT COUPLED ENVELOPE
# ============================================================

exec(
    (CPP_DIR /
        "benchmark_stage_i.py"
    ).read_text(
        encoding="utf-8"
    )
)


Loaded Stage H: C++17
Loaded Stage I: C++17

m=100..1,000 | N=1
Stage H:    0.0263s | accept= 0.037729 | props/draw=   26.505 | M=  3.03431
Stage I:    0.1419s | accept= 0.043794 | props/draw=   22.834 | M_pair=  2.81066
I/H speedup=   0.185 | acceptance gain=   1.161 | mean exact separable-pair slack removed=  0.14855

m=100..1,000 | N=10
Stage H:    0.0826s | accept= 0.044124 | props/draw=   22.664 | M=  2.97965
Stage I:    0.1337s | accept= 0.050245 | props/draw=   19.903 | M_pair=  2.74623
I/H speedup=   0.618 | acceptance gain=   1.139 | mean exact separable-pair slack removed=  0.15865

m=100..1,000 | N=100
Stage H:    0.2649s | accept= 0.035711 | props/draw=   28.002 | M=  3.13385
Stage I:    0.2610s | accept= 0.039996 | props/draw=   25.002 | M_pair=  2.95003
I/H speedup=   1.015 | acceptance gain=   1.120 | mean exact separable-pair slack removed=  0.10725

m=1,000..10,000 | N=1
Stage H:    0.0141s | accept= 0.022100 | props/draw=   45.248 | M=  3.67065
Stage I:    0.2804s | a

In [44]:
import sys
import subprocess
import importlib


# ============================================================
# STAGE K — COMPILE + IMPORT
# ============================================================

subprocess.run(
    [
        sys.executable,
        "setup_stage_k.py",
        "build_ext",
        "--inplace",
        "--force",
    ],
    cwd=CPP_DIR,
    text=True,
    capture_output=True,
    check=True,
)

importlib.invalidate_caches()

import fast_db_stage_h
import fast_db_stage_k


print("Stage H C++:", fast_db_stage_h.cpp_version())
print("Stage K C++:", fast_db_stage_k.cpp_version())

functions = [
    "double_binomial_many_gaussian_fast",
    "double_binomial_many_gaussian_fast_diagnostics",
    "gaussian_envelope_setup_optimized",
]

for name in functions:
    print(
        f"{name}:",
        hasattr(
            fast_db_stage_k,
            name,
        ),
    )

    

Stage H C++: C++17
Stage K C++: C++17
double_binomial_many_gaussian_fast: True
double_binomial_many_gaussian_fast_diagnostics: True
gaussian_envelope_setup_optimized: True


In [45]:
import gc
import time
import numpy as np
import pandas as pd


# ============================================================
# STAGE H K=32 VS STAGE K DIRECT TARGET-GAUSSIAN RNG
# ============================================================

MASTER_SEED = 20260912
REPEATS = 3

H_BOUND_K = 32
K_EXACT_SCAN_THRESHOLD = 64

MAX_PROPOSALS_PER_PARAMETER = 0
TIMEOUT_SECONDS = 300.0


# ============================================================
# WORKLOADS
# ============================================================

WORKLOADS = [
    {
        "workload": "m=10..100 | N=1",
        "m_low": 10,
        "m_high": 100,
        "n_parameters": 50_000,
        "draws_per_parameter": 1,
    },
    {
        "workload": "m=10..100 | N=10",
        "m_low": 10,
        "m_high": 100,
        "n_parameters": 10_000,
        "draws_per_parameter": 10,
    },
    {
        "workload": "m=10..100 | N=100",
        "m_low": 10,
        "m_high": 100,
        "n_parameters": 2_000,
        "draws_per_parameter": 100,
    },
    {
        "workload": "m=100..1,000 | N=1",
        "m_low": 100,
        "m_high": 1_000,
        "n_parameters": 20_000,
        "draws_per_parameter": 1,
    },
    {
        "workload": "m=100..1,000 | N=10",
        "m_low": 100,
        "m_high": 1_000,
        "n_parameters": 5_000,
        "draws_per_parameter": 10,
    },
    {
        "workload": "m=1,000..10,000 | N=1",
        "m_low": 1_000,
        "m_high": 10_000,
        "n_parameters": 5_000,
        "draws_per_parameter": 1,
    },
    {
        "workload": "m=1,000..10,000 | N=10",
        "m_low": 1_000,
        "m_high": 10_000,
        "n_parameters": 1_000,
        "draws_per_parameter": 10,
    },
    {
        "workload": "m=10,000..100,000 | N=1",
        "m_low": 10_000,
        "m_high": 100_000,
        "n_parameters": 1_000,
        "draws_per_parameter": 1,
    },
]


# ============================================================
# PARAMETER GENERATION
# ============================================================

def logistic(x):
    return 1.0 / (1.0 + np.exp(-x))


def make_parameters(
    n_parameters,
    m_low,
    m_high,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    m_values = rng.integers(
        m_low,
        m_high + 1,
        size=n_parameters,
        dtype=np.int64,
    )

    logit_p = rng.uniform(
        -8.0,
        8.0,
        size=n_parameters,
    )

    p_values = logistic(
        logit_p
    ).astype(
        np.float64
    )

    theta_values = rng.uniform(
        0.05,
        0.95,
        size=n_parameters,
    ).astype(
        np.float64
    )

    return (
        np.ascontiguousarray(
            m_values
        ),
        np.ascontiguousarray(
            p_values
        ),
        np.ascontiguousarray(
            theta_values
        ),
    )


# ============================================================
# RUN BENCHMARK
# ============================================================

rows = []

print()
print("=" * 116)
print(
    "STAGE H K=32 VS "
    "STAGE K DIRECT TARGET-GAUSSIAN RNG"
)
print("=" * 116)


for workload_index, spec in enumerate(
    WORKLOADS
):

    label = spec[
        "workload"
    ]

    n_parameters = spec[
        "n_parameters"
    ]

    draws_per_parameter = spec[
        "draws_per_parameter"
    ]

    total_draws = (
        n_parameters
        * draws_per_parameter
    )

    (
        m_values,
        p_values,
        theta_values,
    ) = make_parameters(
        n_parameters=n_parameters,
        m_low=spec["m_low"],
        m_high=spec["m_high"],
        seed=(
            MASTER_SEED
            + workload_index
        ),
    )


    # --------------------------------------------------------
    # WARM-UP
    # --------------------------------------------------------

    warm_n = min(
        64,
        n_parameters,
    )

    fast_db_stage_h \
        .double_binomial_many_fast_constant_optimized(
            m_values[
                :warm_n
            ],
            p_values[
                :warm_n
            ],
            theta_values[
                :warm_n
            ],
            draws_per_parameter=1,
            seed=12345,
            bound_K=H_BOUND_K,
            max_proposals_per_parameter=0,
            timeout_seconds=TIMEOUT_SECONDS,
        )

    fast_db_stage_k \
        .double_binomial_many_gaussian_fast(
            m_values[
                :warm_n
            ],
            p_values[
                :warm_n
            ],
            theta_values[
                :warm_n
            ],
            draws_per_parameter=1,
            seed=12345,
            exact_scan_threshold=(
                K_EXACT_SCAN_THRESHOLD
            ),
            max_proposals_per_parameter=0,
            timeout_seconds=TIMEOUT_SECONDS,
        )


    # --------------------------------------------------------
    # COMMON ARGUMENTS
    # --------------------------------------------------------

    common_h = dict(
        m_values=m_values,
        p_values=p_values,
        theta_values=theta_values,

        draws_per_parameter=(
            draws_per_parameter
        ),

        bound_K=H_BOUND_K,

        max_proposals_per_parameter=(
            MAX_PROPOSALS_PER_PARAMETER
        ),

        timeout_seconds=(
            TIMEOUT_SECONDS
        ),
    )

    common_k = dict(
        m_values=m_values,
        p_values=p_values,
        theta_values=theta_values,

        draws_per_parameter=(
            draws_per_parameter
        ),

        exact_scan_threshold=(
            K_EXACT_SCAN_THRESHOLD
        ),

        max_proposals_per_parameter=(
            MAX_PROPOSALS_PER_PARAMETER
        ),

        timeout_seconds=(
            TIMEOUT_SECONDS
        ),
    )


    # --------------------------------------------------------
    # TIMING
    #
    # Use non-diagnostic functions.
    # Alternate method order across repetitions.
    # --------------------------------------------------------

    h_times = []
    k_times = []

    for repeat in range(
        REPEATS
    ):

        gc.collect()

        method_order = (
            ("H", "K")
            if repeat % 2 == 0
            else ("K", "H")
        )

        for method in method_order:

            seed = (
                700_000
                + repeat
            )

            if method == "H":

                start = (
                    time.perf_counter()
                )

                fast_db_stage_h \
                    .double_binomial_many_fast_constant_optimized(
                        **common_h,
                        seed=seed,
                    )

                h_times.append(
                    time.perf_counter()
                    - start
                )

            else:

                start = (
                    time.perf_counter()
                )

                fast_db_stage_k \
                    .double_binomial_many_gaussian_fast(
                        **common_k,
                        seed=seed,
                    )

                k_times.append(
                    time.perf_counter()
                    - start
                )


    h_seconds = float(
        np.median(
            h_times
        )
    )

    k_seconds = float(
        np.median(
            k_times
        )
    )


    # --------------------------------------------------------
    # DIAGNOSTICS
    #
    # Separate from timing.
    # Same parameter arrays.
    # --------------------------------------------------------

    diag_seed = (
        900_000
        + workload_index
    )

    h_diag = (
        fast_db_stage_h
        .double_binomial_many_fast_constant_optimized_diagnostics(
            **common_h,
            seed=diag_seed,
        )
    )

    k_diag = (
        fast_db_stage_k
        .double_binomial_many_gaussian_fast_diagnostics(
            **common_k,
            seed=diag_seed,
        )
    )


    # --------------------------------------------------------
    # SUMMARIZE
    # --------------------------------------------------------

    h_accept = float(
        h_diag[
            "acceptance"
        ]
    )

    k_accept = float(
        k_diag[
            "acceptance"
        ]
    )

    h_proposals_per_draw = (
        int(
            h_diag[
                "proposals"
            ]
        )
        / total_draws
    )

    k_proposals_per_draw = (
        int(
            k_diag[
                "proposals"
            ]
        )
        / total_draws
    )

    row = {
        "workload": label,

        "n_parameters": (
            n_parameters
        ),

        "draws_per_parameter": (
            draws_per_parameter
        ),

        "total_draws": (
            total_draws
        ),

        "H_seconds": (
            h_seconds
        ),

        "K_seconds": (
            k_seconds
        ),

        "K_speedup_vs_H": (
            h_seconds
            / k_seconds
        ),

        "H_draws_per_second": (
            total_draws
            / h_seconds
        ),

        "K_draws_per_second": (
            total_draws
            / k_seconds
        ),

        "H_acceptance": (
            h_accept
        ),

        "K_acceptance": (
            k_accept
        ),

        "K_acceptance_ratio_vs_H": (
            k_accept
            / h_accept
        ),

        "H_proposals_per_draw": (
            h_proposals_per_draw
        ),

        "K_proposals_per_draw": (
            k_proposals_per_draw
        ),

        "H_mean_U_target": float(
            h_diag[
                "mean_U_target"
            ]
        ),

        "K_mean_U_target": float(
            k_diag[
                "mean_U_target"
            ]
        ),

        "H_mean_target_queries": float(
            h_diag[
                "mean_target_exact_queries_per_setup"
            ]
        ),

        "K_mean_target_queries": float(
            k_diag[
                "mean_target_exact_queries_per_setup"
            ]
        ),

        "H_max_log_accept": float(
            h_diag[
                "maximum_log_accept"
            ]
        ),

        "K_max_log_accept": float(
            k_diag[
                "maximum_log_accept"
            ]
        ),
    }

    rows.append(
        row
    )


    print()
    print(
        label
    )

    print(
        f"  H: "
        f"{h_seconds:9.5f}s | "
        f"accept={h_accept:9.6f} | "
        f"props/draw={h_proposals_per_draw:9.3f}"
    )

    print(
        f"  K: "
        f"{k_seconds:9.5f}s | "
        f"accept={k_accept:9.6f} | "
        f"props/draw={k_proposals_per_draw:9.3f}"
    )

    print(
        f"  K/H speedup="
        f"{h_seconds / k_seconds:9.4f} | "
        f"acceptance ratio="
        f"{k_accept / h_accept:9.4f}"
    )


# ============================================================
# RESULTS TABLE
# ============================================================

results = pd.DataFrame(
    rows
)

print()
print()
print("=" * 135)
print("COMBINED RESULTS")
print("=" * 135)

print(
    results[
        [
            "workload",

            "H_seconds",
            "K_seconds",
            "K_speedup_vs_H",

            "H_acceptance",
            "K_acceptance",
            "K_acceptance_ratio_vs_H",

            "H_proposals_per_draw",
            "K_proposals_per_draw",

            "H_mean_U_target",
            "K_mean_U_target",

            "H_mean_target_queries",
            "K_mean_target_queries",

            "H_max_log_accept",
            "K_max_log_accept",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# SHORT DECISION TABLE
# ============================================================

print()
print("=" * 135)
print("SPEED / ACCEPTANCE SUMMARY")
print("=" * 135)

print(
    results[
        [
            "workload",
            "K_speedup_vs_H",
            "K_acceptance_ratio_vs_H",
            "H_proposals_per_draw",
            "K_proposals_per_draw",
        ]
    ].to_string(
        index=False
    )
)


print()
print(
    "Largest observed "
    "Stage-K log acceptance:",
    results[
        "K_max_log_accept"
    ].max(),
)


STAGE H K=32 VS STAGE K DIRECT TARGET-GAUSSIAN RNG

m=10..100 | N=1
  H:   0.90020s | accept= 0.082250 | props/draw=   12.158
  K:   0.23827s | accept= 0.343492 | props/draw=    2.911
  K/H speedup=   3.7781 | acceptance ratio=   4.1762

m=10..100 | N=10
  H:   0.69232s | accept= 0.082776 | props/draw=   12.081
  K:   0.12868s | accept= 0.343138 | props/draw=    2.914
  K/H speedup=   5.3802 | acceptance ratio=   4.1454

m=10..100 | N=100
  H:   1.08808s | accept= 0.087052 | props/draw=   11.487
  K:   0.16935s | accept= 0.347867 | props/draw=    2.875
  K/H speedup=   6.4249 | acceptance ratio=   3.9961

m=100..1,000 | N=1
  H:   0.51264s | accept= 0.041313 | props/draw=   24.205
  K:   0.05143s | accept= 0.185491 | props/draw=    5.391
  K/H speedup=   9.9675 | acceptance ratio=   4.4899

m=100..1,000 | N=10
  H:   0.85909s | accept= 0.040562 | props/draw=   24.653
  K:   0.10992s | accept= 0.182343 | props/draw=    5.484
  K/H speedup=   7.8154 | acceptance ratio=   4.4954

m=1,000

In [46]:
import gc
import time
import numpy as np
import pandas as pd


# ============================================================
# STAGE K DIRECT GAUSSIAN
# VS
# OPTIMIZED NATIVE PMF
#
# Requires already imported:
#
#     fast_db_stage_k
#     fast_db_stage_h
#
# Stage H contains:
#     double_binomial_many_pmf_optimized
#
# Stage K contains:
#     double_binomial_many_gaussian_fast
#     double_binomial_many_gaussian_fast_diagnostics
# ============================================================


MASTER_SEED = 20260912
REPEATS = 3

K_EXACT_SCAN_THRESHOLD = 64

MAX_PROPOSALS_PER_PARAMETER = 0
TIMEOUT_SECONDS = 300.0


# ============================================================
# WORKLOADS
#
# Same general workload structure used previously.
# ============================================================

WORKLOADS = [
    {
        "workload": "m=10..100 | N=1",
        "m_low": 10,
        "m_high": 100,
        "n_parameters": 50_000,
        "draws_per_parameter": 1,
    },
    {
        "workload": "m=10..100 | N=10",
        "m_low": 10,
        "m_high": 100,
        "n_parameters": 10_000,
        "draws_per_parameter": 10,
    },
    {
        "workload": "m=10..100 | N=100",
        "m_low": 10,
        "m_high": 100,
        "n_parameters": 2_000,
        "draws_per_parameter": 100,
    },
    {
        "workload": "m=100..1,000 | N=1",
        "m_low": 100,
        "m_high": 1_000,
        "n_parameters": 20_000,
        "draws_per_parameter": 1,
    },
    {
        "workload": "m=100..1,000 | N=10",
        "m_low": 100,
        "m_high": 1_000,
        "n_parameters": 5_000,
        "draws_per_parameter": 10,
    },
    {
        "workload": "m=1,000..10,000 | N=1",
        "m_low": 1_000,
        "m_high": 10_000,
        "n_parameters": 5_000,
        "draws_per_parameter": 1,
    },
    {
        "workload": "m=1,000..10,000 | N=10",
        "m_low": 1_000,
        "m_high": 10_000,
        "n_parameters": 1_000,
        "draws_per_parameter": 10,
    },
    {
        "workload": "m=10,000..100,000 | N=1",
        "m_low": 10_000,
        "m_high": 100_000,
        "n_parameters": 1_000,
        "draws_per_parameter": 1,
    },
]


# ============================================================
# PARAMETER GENERATION
#
# Same distribution used in the recent benchmarks:
#
#     m        integer uniform over workload range
#     logit p  uniform(-8, 8)
#     theta    uniform(.05, .95)
# ============================================================

def logistic(x):
    return 1.0 / (1.0 + np.exp(-x))


def make_parameters(
    n_parameters,
    m_low,
    m_high,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    m_values = rng.integers(
        m_low,
        m_high + 1,
        size=n_parameters,
        dtype=np.int64,
    )

    logit_p = rng.uniform(
        -8.0,
        8.0,
        size=n_parameters,
    )

    p_values = logistic(
        logit_p
    ).astype(
        np.float64
    )

    theta_values = rng.uniform(
        0.05,
        0.95,
        size=n_parameters,
    ).astype(
        np.float64
    )

    return (
        np.ascontiguousarray(
            m_values
        ),
        np.ascontiguousarray(
            p_values
        ),
        np.ascontiguousarray(
            theta_values
        ),
    )


# ============================================================
# BENCHMARK
# ============================================================

rows = []

print()
print("=" * 120)
print(
    "STAGE K DIRECT TARGET-GAUSSIAN "
    "VS OPTIMIZED NATIVE PMF"
)
print("=" * 120)


for workload_index, spec in enumerate(
    WORKLOADS
):

    label = spec[
        "workload"
    ]

    n_parameters = spec[
        "n_parameters"
    ]

    draws_per_parameter = spec[
        "draws_per_parameter"
    ]

    total_draws = (
        n_parameters
        * draws_per_parameter
    )


    # --------------------------------------------------------
    # SAME PARAMETER TRIPLES FOR BOTH METHODS
    # --------------------------------------------------------

    (
        m_values,
        p_values,
        theta_values,
    ) = make_parameters(
        n_parameters=n_parameters,
        m_low=spec["m_low"],
        m_high=spec["m_high"],
        seed=(
            MASTER_SEED
            + workload_index
        ),
    )


    # ========================================================
    # WARM-UP
    # ========================================================

    warm_n = min(
        64,
        n_parameters,
    )

    fast_db_stage_k \
        .double_binomial_many_gaussian_fast(
            m_values[:warm_n],
            p_values[:warm_n],
            theta_values[:warm_n],
            draws_per_parameter=1,
            seed=12345,
            exact_scan_threshold=(
                K_EXACT_SCAN_THRESHOLD
            ),
            max_proposals_per_parameter=0,
            timeout_seconds=TIMEOUT_SECONDS,
        )

    fast_db_stage_h \
        .double_binomial_many_pmf_optimized(
            m_values[:warm_n],
            p_values[:warm_n],
            theta_values[:warm_n],
            draws_per_parameter=1,
            seed=12345,
            timeout_seconds=TIMEOUT_SECONDS,
        )


    # ========================================================
    # COMMON CALL ARGUMENTS
    # ========================================================

    common_k = dict(
        m_values=m_values,
        p_values=p_values,
        theta_values=theta_values,

        draws_per_parameter=(
            draws_per_parameter
        ),

        exact_scan_threshold=(
            K_EXACT_SCAN_THRESHOLD
        ),

        max_proposals_per_parameter=(
            MAX_PROPOSALS_PER_PARAMETER
        ),

        timeout_seconds=(
            TIMEOUT_SECONDS
        ),
    )

    common_pmf = dict(
        m_values=m_values,
        p_values=p_values,
        theta_values=theta_values,

        draws_per_parameter=(
            draws_per_parameter
        ),

        timeout_seconds=(
            TIMEOUT_SECONDS
        ),
    )


    # ========================================================
    # TIMING
    #
    # Alternate method order across repeats so one method does
    # not always benefit from going second.
    # ========================================================

    k_times = []
    pmf_times = []

    for repeat in range(
        REPEATS
    ):

        gc.collect()

        order = (
            ("K", "PMF")
            if repeat % 2 == 0
            else ("PMF", "K")
        )

        for method in order:

            seed = (
                700_000
                + repeat
            )

            if method == "K":

                start = (
                    time.perf_counter()
                )

                fast_db_stage_k \
                    .double_binomial_many_gaussian_fast(
                        **common_k,
                        seed=seed,
                    )

                elapsed = (
                    time.perf_counter()
                    - start
                )

                k_times.append(
                    elapsed
                )

            else:

                start = (
                    time.perf_counter()
                )

                fast_db_stage_h \
                    .double_binomial_many_pmf_optimized(
                        **common_pmf,
                        seed=seed,
                    )

                elapsed = (
                    time.perf_counter()
                    - start
                )

                pmf_times.append(
                    elapsed
                )


    # ========================================================
    # MEDIAN TIMINGS
    # ========================================================

    k_seconds = float(
        np.median(
            k_times
        )
    )

    pmf_seconds = float(
        np.median(
            pmf_times
        )
    )


    # ========================================================
    # STAGE K DIAGNOSTICS
    #
    # Run separately so diagnostics do not contaminate timing.
    # ========================================================

    k_diag = (
        fast_db_stage_k
        .double_binomial_many_gaussian_fast_diagnostics(
            **common_k,
            seed=(
                900_000
                + workload_index
            ),
        )
    )

    k_acceptance = float(
        k_diag[
            "acceptance"
        ]
    )

    k_proposals = int(
        k_diag[
            "proposals"
        ]
    )

    k_proposals_per_draw = (
        k_proposals
        / total_draws
    )

    k_max_log_accept = float(
        k_diag[
            "maximum_log_accept"
        ]
    )

    k_mean_U_target = float(
        k_diag[
            "mean_U_target"
        ]
    )

    k_mean_target_queries = float(
        k_diag[
            "mean_target_exact_queries_per_setup"
        ]
    )


    # ========================================================
    # RESULT ROW
    # ========================================================

    row = {
        "workload": label,

        "n_parameters": (
            n_parameters
        ),

        "draws_per_parameter": (
            draws_per_parameter
        ),

        "total_draws": (
            total_draws
        ),

        "K_seconds": (
            k_seconds
        ),

        "PMF_seconds": (
            pmf_seconds
        ),

        # > 1 means Stage K is faster.
        "K_speedup_vs_PMF": (
            pmf_seconds
            / k_seconds
        ),

        "K_draws_per_second": (
            total_draws
            / k_seconds
        ),

        "PMF_draws_per_second": (
            total_draws
            / pmf_seconds
        ),

        "K_acceptance": (
            k_acceptance
        ),

        "K_proposals_per_draw": (
            k_proposals_per_draw
        ),

        "K_mean_U_target": (
            k_mean_U_target
        ),

        "K_mean_target_queries": (
            k_mean_target_queries
        ),

        "K_max_log_accept": (
            k_max_log_accept
        ),

        "K_min_seconds": float(
            np.min(
                k_times
            )
        ),

        "K_max_seconds": float(
            np.max(
                k_times
            )
        ),

        "PMF_min_seconds": float(
            np.min(
                pmf_times
            )
        ),

        "PMF_max_seconds": float(
            np.max(
                pmf_times
            )
        ),
    }

    rows.append(
        row
    )


    # ========================================================
    # LIVE OUTPUT
    # ========================================================

    print()
    print(label)

    print(
        f"  Stage K | "
        f"time={k_seconds:9.5f}s | "
        f"draws/s={total_draws / k_seconds:12.3f} | "
        f"accept={k_acceptance:9.6f} | "
        f"props/draw={k_proposals_per_draw:8.3f}"
    )

    print(
        f"  PMF     | "
        f"time={pmf_seconds:9.5f}s | "
        f"draws/s={total_draws / pmf_seconds:12.3f}"
    )

    print(
        f"  K / PMF speedup = "
        f"{pmf_seconds / k_seconds:9.4f}x"
    )


# ============================================================
# COMBINED RESULTS
# ============================================================

results = pd.DataFrame(
    rows
)

print()
print()
print("=" * 145)
print("COMBINED RESULTS")
print("=" * 145)

print(
    results[
        [
            "workload",

            "K_seconds",
            "PMF_seconds",
            "K_speedup_vs_PMF",

            "K_draws_per_second",
            "PMF_draws_per_second",

            "K_acceptance",
            "K_proposals_per_draw",

            "K_mean_U_target",
            "K_mean_target_queries",

            "K_max_log_accept",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# CROSSOVER SUMMARY
# ============================================================

print()
print("=" * 145)
print("STAGE K VS PMF CROSSOVER SUMMARY")
print("=" * 145)

summary = results[
    [
        "workload",
        "K_speedup_vs_PMF",
        "K_acceptance",
        "K_proposals_per_draw",
    ]
].copy()

summary[
    "winner"
] = np.where(
    summary[
        "K_speedup_vs_PMF"
    ] > 1.0,
    "Stage K",
    "PMF",
)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# SIMPLE WIN COUNTS
# ============================================================

n_k_wins = int(
    (
        results[
            "K_speedup_vs_PMF"
        ]
        > 1.0
    ).sum()
)

n_pmf_wins = (
    len(results)
    - n_k_wins
)

print()
print(
    f"Stage K wins: {n_k_wins}"
)

print(
    f"PMF wins:     {n_pmf_wins}"
)

print()
print(
    "Largest observed "
    "Stage-K log acceptance:",
    results[
        "K_max_log_accept"
    ].max(),
)



STAGE K DIRECT TARGET-GAUSSIAN VS OPTIMIZED NATIVE PMF

m=10..100 | N=1
  Stage K | time=  0.25789s | draws/s=  193879.082 | accept= 0.343492 | props/draw=   2.911
  PMF     | time=  0.03575s | draws/s= 1398472.306
  K / PMF speedup =    0.1386x

m=10..100 | N=10
  Stage K | time=  0.12634s | draws/s=  791523.104 | accept= 0.343138 | props/draw=   2.914
  PMF     | time=  0.00934s | draws/s=10711799.098
  K / PMF speedup =    0.0739x

m=10..100 | N=100
  Stage K | time=  0.18565s | draws/s= 1077300.629 | accept= 0.347867 | props/draw=   2.875
  PMF     | time=  0.00768s | draws/s=26049128.582
  K / PMF speedup =    0.0414x

m=100..1,000 | N=1
  Stage K | time=  0.05373s | draws/s=  372228.757 | accept= 0.185491 | props/draw=   5.391
  PMF     | time=  0.16364s | draws/s=  122216.369
  K / PMF speedup =    3.0457x

m=100..1,000 | N=10
  Stage K | time=  0.10751s | draws/s=  465089.455 | accept= 0.182343 | props/draw=   5.484
  PMF     | time=  0.04223s | draws/s= 1184037.283
  K / PMF 